# ViT Full Design FPGA Control Notebook

這份 notebook 對應 RTL top：`ViT_DDR_AxiLite_Wrapper.sv`。

Python 只負責配置 DDR buffer、寫 AXI-Lite descriptor、輪詢 status/counter；真正把 DDR 資料搬到 RTL loader 的動作由 `ViT_DDR_AxiLite_Wrapper` 的 M_AXI master 完成。

目前 DDR master 是 32-bit single-beat bring-up 版本。後續若效能不足，可在 RTL 端把 LOAD/STORE FSM 改成 AXI burst，Python register map 不需要大改。


In [1]:
from pathlib import Path
import json
import math
import re
import time
import numpy as np

try:
    from pynq import Overlay, MMIO, allocate
except ImportError:
    Overlay = None
    MMIO = None
    allocate = None

DRIVER_VERSION = "2026-06-25-stage-checkpoint-v18-rtl-physical-golden"

NOTEBOOK_DIR = Path.cwd()
PRINT_ACCURACY_METRICS = False  # Set True only when you want golden/accuracy mismatch details.
BITSTREAM = NOTEBOOK_DIR / "vit_fulldesign.bit"
ENABLE_AXI_LITE_FALLBACK = False  # Official performance: use RTL AXI master, not AXI-Lite data fallback.
IP_NAME_CANDIDATES = [
    "ViT_DDR_AxiLite_Wrap_0",
    "ViT_DDR_AxiLite_Wrap_0_0",
    "ViT_DDR_AxiLite_Wrapper_0",
    "ViT_DDR_AxiLite_Wrapper_0_0",
    "vit_bd/ViT_DDR_AxiLite_Wrap_0",
    "vit_bd/ViT_DDR_AxiLite_Wrapper_0",
    "vit_ddr_axilite_wrapper_0",
    "vit_ddr_0",
]

# Golden data on the Windows host is generated by:
#   C:\Users\angelliu.LAPTOP-3NTJHQPG\Downloads\golden_gen\golden_gen.py
# This notebook intentionally uses only case_vit_real_model for the one-block
# accuracy test, so it will not silently fall back to old smoke vectors.
GOLDEN_CASE_NAME = "case_vit_real_model"
HOST_GOLDEN_ROOT = Path(r"C:\Users\angelliu.LAPTOP-3NTJHQPG\Downloads\golden_gen")
GOLDEN_ROOT_CANDIDATES = [
    NOTEBOOK_DIR / "golden_gen" / "hex" / GOLDEN_CASE_NAME,
    HOST_GOLDEN_ROOT / "hex" / GOLDEN_CASE_NAME,
]
GOLDEN_ROOT = next((p for p in GOLDEN_ROOT_CANDIDATES if p.exists()), GOLDEN_ROOT_CANDIDATES[0])
if not GOLDEN_ROOT.exists():
    print("WARNING: real-model vectors were not found. Run golden_gen.py first or copy case_vit_real_model next to this notebook.")
REPORT_DIR = NOTEBOOK_DIR / "reports"

print("Notebook driver:", DRIVER_VERSION)
print("Notebook dir:", NOTEBOOK_DIR)
print("Selected golden root:", GOLDEN_ROOT)


Notebook driver: 2026-06-25-stage-checkpoint-v18-rtl-physical-golden
Notebook dir: /home/xilinx/jupyter_notebooks/VIT_fulldesign
Selected golden root: /home/xilinx/jupyter_notebooks/VIT_fulldesign/golden_gen/hex/case_vit_real_model


## Register Map

這些 offset 需要和 `ViT_DDR_AxiLite_Wrapper.sv` 保持一致。


In [2]:
DRIVER_VERSION = "2026-06-25-stage-checkpoint-v18-rtl-physical-golden"
ENABLE_AXI_LITE_FALLBACK = False

REG_CONTROL      = 0x000
REG_PATCH_SHIFT  = 0x004
REG_STATUS       = 0x008
REG_DEBUG        = 0x00C
REG_PATCH_REQ    = 0x010
REG_TRANS_REQ    = 0x014
REG_INPUT_TGT    = 0x018
REG_INPUT_COUNT  = 0x01C

REG_DDR_SRC_ADDR = 0x020
REG_DDR_DST_ADDR = 0x024
REG_DDR_BASE     = 0x028
REG_DDR_COUNT    = 0x02C
REG_DDR_TARGET   = 0x030
REG_DDR_STATUS   = 0x034
REG_DDR_INDEX    = 0x038
REG_DDR_CONTROL  = 0x03C

REG_PERF_CONTROL = 0x040
REG_PERF_TOTAL   = 0x044
REG_PERF_BUSY    = 0x048
REG_PERF_DDR_LD  = 0x04C
REG_PERF_DDR_ST  = 0x050
REG_PERF_OVERLAP = 0x054
REG_PERF_STALL   = 0x058
REG_PERF_DDR_RDW = 0x05C
REG_PERF_DDR_WRW = 0x060
REG_PERF_LDR_WR  = 0x064
REG_PERF_OUT_WR  = 0x068
REG_PERF_HOST_WR = 0x06C
REG_PERF_OUT_RD  = 0x070
REG_PERF_ACTIVE  = 0x074
REG_PAGE_REQ     = 0x078
REG_PERF_BRAM_RDW = 0x07C
REG_PERF_BRAM_WRW = 0x080
REG_PERF_BRAM_CYC = 0x084
REG_PERF_MAC_LO   = 0x088
REG_PERF_MAC_HI   = 0x08C
REG_PERF_PP_WAIT  = 0x090
REG_PERF_PP_LOAD  = 0x094
REG_PERF_PP_OVLP  = 0x098
REG_RUN_MODE       = 0x09C   # bit0=transformer-only, skip patch embedding
REG_PERF_SYS_RUN  = 0x0E0   # systolic cs != IDLE cycles
REG_PERF_SYS_OP   = 0x0E4   # systolic OP-state cycles
REG_PERF_SYS_STEP = 0x0E8   # systolic row/psum issue steps (pass pulses)
REG_PERF_DDR_RTXN = 0x110   # AXI read address transactions
REG_PERF_DDR_WTXN = 0x114   # AXI write address transactions

INPUT_BASE       = 0x100
INPUT_CTRL       = INPUT_BASE + 0x0
INPUT_BASE_REG   = INPUT_BASE + 0x4
INPUT_COUNT_REG  = INPUT_BASE + 0x8
INPUT_DATA_REG   = INPUT_BASE + 0xC
OUTPUT_BASE      = 0x1000

TARGET_IMAGE     = 0
TARGET_POS       = 1
TARGET_CLS       = 2
TARGET_INV_RMS   = 3   # legacy no-op in current RTL
TARGET_GAMMA     = 4   # one signed int16 gamma in DATA[15:0] per loader word
TARGET_PATCH_W   = 5
TARGET_PATCH_B   = 6
TARGET_TRANS_W   = 7
TARGET_TRANS_B   = 8
TARGET_GELU_PAGE = 9   # 1024-word GELU_out page cache, load/store through PAGE_REQ
TARGET_X_BUF     = 10  # packed X buffer words for Python-driven multi-block replay

PERF_REGS = {
    "total_cycles": REG_PERF_TOTAL,
    "compute_busy_cycles": REG_PERF_BUSY,
    "ddr_load_cycles": REG_PERF_DDR_LD,
    "ddr_store_cycles": REG_PERF_DDR_ST,
    "overlap_cycles": REG_PERF_OVERLAP,
    "tile_stall_cycles": REG_PERF_STALL,
    "ddr_read_words": REG_PERF_DDR_RDW,
    "ddr_write_words": REG_PERF_DDR_WRW,
    "ddr_read_transactions": REG_PERF_DDR_RTXN,
    "ddr_write_transactions": REG_PERF_DDR_WTXN,
    "loader_data_words": REG_PERF_LDR_WR,
    "output_store_words": REG_PERF_OUT_WR,
    "host_loader_words": REG_PERF_HOST_WR,
    "output_read_cycles": REG_PERF_OUT_RD,
    "active_cycles": REG_PERF_ACTIVE,
    "bram_read_words": REG_PERF_BRAM_RDW,
    "bram_write_words": REG_PERF_BRAM_WRW,
    "bram_access_cycles": REG_PERF_BRAM_CYC,
    "mac_ops_lo": REG_PERF_MAC_LO,
    "mac_ops_hi": REG_PERF_MAC_HI,
    "pingpong_wait_cycles": REG_PERF_PP_WAIT,
    "pingpong_load_cycles": REG_PERF_PP_LOAD,
    "pingpong_overlap_cycles": REG_PERF_PP_OVLP,
    "systolic_run_cycles": REG_PERF_SYS_RUN,
    "systolic_op_cycles": REG_PERF_SYS_OP,
    "systolic_pe_step_cycles": REG_PERF_SYS_STEP,
}


## Overlay / MMIO Setup

若 IP 名稱和 block design 裡不同，修改 `IP_NAME_CANDIDATES` 或直接指定 `ip_base`。


In [3]:
def load_overlay(bitstream=BITSTREAM, ip_name_candidates=IP_NAME_CANDIDATES, ip_base=None, addr_range=0x2000):
    if Overlay is None:
        raise RuntimeError("pynq is not available in this Python environment")
    ol = Overlay(str(bitstream))
    if ip_base is None:
        ip_dict = ol.ip_dict
        chosen = None
        for name in ip_name_candidates:
            if name in ip_dict:
                chosen = name
                break
        if chosen is None:
            print("Available IP names:")
            for name in ip_dict:
                print("  ", name)
            raise KeyError("Cannot find ViT DDR AXI-Lite IP; update IP_NAME_CANDIDATES")
        base = ip_dict[chosen]["phys_addr"]
        rng = ip_dict[chosen].get("addr_range", addr_range)
        print(f"Using IP {chosen}: base=0x{base:08x}, range=0x{rng:x}")
        return ol, MMIO(base, rng)
    print(f"Using manual MMIO base=0x{ip_base:08x}, range=0x{addr_range:x}")
    return ol, MMIO(ip_base, addr_range)

# Example:
# overlay, mmio = load_overlay()


## MMIO Helpers


In [4]:
def wr(mmio, offset, value):
    mmio.write(offset, int(value) & 0xFFFFFFFF)


def rd(mmio, offset):
    return int(mmio.read(offset)) & 0xFFFFFFFF


def pulse(mmio, offset, mask):
    wr(mmio, offset, mask)


def decode_status(status):
    return {
        "busy": bool(status & (1 << 0)),
        "done_sticky": bool(status & (1 << 2)),
        "input_busy": bool(status & (1 << 3)),
        "input_done": bool(status & (1 << 4)),
        "input_error": bool(status & (1 << 5)),
        "patch_weight_miss": bool(status & (1 << 6)),
        "patch_bias_miss": bool(status & (1 << 7)),
        "trans_weight_miss": bool(status & (1 << 8)),
        "trans_bias_miss": bool(status & (1 << 9)),
        "patch_weight_valid": bool(status & (1 << 10)),
        "patch_bias_valid": bool(status & (1 << 11)),
        "trans_weight_valid": bool(status & (1 << 12)),
        "trans_bias_valid": bool(status & (1 << 13)),
        "patch_weight_match": bool(status & (1 << 14)),
        "patch_bias_match": bool(status & (1 << 15)),
        "trans_weight_match": bool(status & (1 << 16)),
        "trans_bias_match": bool(status & (1 << 17)),
        "trans_external_phase": bool(status & (1 << 18)),
        "patch_external_phase": bool(status & (1 << 19)),
    }



DDR_LOAD_STATES = {
    0: "LOAD_IDLE",
    1: "LOAD_CFG_BASE",
    2: "LOAD_CFG_COUNT",
    3: "LOAD_CFG_CTRL",
    4: "LOAD_AR",
    5: "LOAD_R",
    6: "LOAD_PUSH",
    7: "LOAD_DONE",
    8: "LOAD_ERROR",
}

DDR_STORE_STATES = {
    0: "STORE_IDLE",
    1: "STORE_REQ",
    2: "STORE_WAIT1",
    3: "STORE_WAIT2",
    4: "STORE_AW_W",
    5: "STORE_B",
    6: "STORE_DONE",
    7: "STORE_ERROR",
}

def decode_debug_word(debug):
    # RTL packs patch debug while patch embedding is busy; otherwise it packs transformer phase.
    return {
        "raw_hex": f"0x{int(debug) & 0xFFFFFFFF:08x}",
        "top_state": int(debug) & 0x3,
        "phase_or_patch_state": (int(debug) >> 8) & 0x1F,
        "tile_or_patch_elem_low8": (int(debug) >> 16) & 0xFF,
        "softmax_or_patch_idx_low8": (int(debug) >> 24) & 0xFF,
    }


def decode_ddr_status(status):
    return {
        "load_error_state": bool(status & (1 << 0)),
        "store_error_state": bool(status & (1 << 1)),
        "load_busy": bool(status & (1 << 4)),
        "store_busy": bool(status & (1 << 5)),
        "load_done": bool(status & (1 << 8)),
        "store_done": bool(status & (1 << 9)),
        "load_error": bool(status & (1 << 10)),
        "store_error": bool(status & (1 << 11)),
        "cmd_error": bool(status & (1 << 12)),
        "load_state": (status >> 28) & 0xF,
        "load_state_name": DDR_LOAD_STATES.get((status >> 28) & 0xF, "UNKNOWN"),
        "store_state": (status >> 24) & 0xF,
        "store_state_name": DDR_STORE_STATES.get((status >> 24) & 0xF, "UNKNOWN"),
    }


def decode_page_req(value):
    value = int(value) & 0xFFFFFFFF
    return {
        "raw": value,
        "valid": bool(value & (1 << 31)),
        "store": bool(value & (1 << 30)),
        "target": (value >> 24) & 0xF,
        "base": value & 0x00FFFFFF,
    }


def wait_until(predicate, timeout_s=10.0, poll_s=0.001, label="condition"):
    t0 = time.time()
    while True:
        value = predicate()
        if value:
            return value
        if time.time() - t0 > timeout_s:
            raise TimeoutError(f"Timeout while waiting for {label}")
        time.sleep(poll_s)


def clear_done(mmio):
    pulse(mmio, REG_CONTROL, 1 << 1)


def clear_ddr_flags(mmio, timeout_s=2.0):
    pulse(mmio, REG_DDR_CONTROL, 1 << 8)

    def flags_clear():
        s = rd(mmio, REG_DDR_STATUS)
        d = decode_ddr_status(s)
        return not (d["load_done"] or d["store_done"] or d["load_error"] or
                    d["store_error"] or d["cmd_error"] or
                    d["load_error_state"] or d["store_error_state"])

    try:
        return wait_until(flags_clear, timeout_s=timeout_s, poll_s=0.0002, label="DDR flags clear")
    except TimeoutError:
        # Try one more clear pulse before reporting the sticky status.
        pulse(mmio, REG_DDR_CONTROL, 1 << 8)
        return wait_until(flags_clear, timeout_s=timeout_s, poll_s=0.0002, label="DDR flags clear retry")


def clear_perf(mmio):
    pulse(mmio, REG_PERF_CONTROL, 1)


def start_compute(mmio):
    pulse(mmio, REG_CONTROL, 1)


def wait_compute_done(mmio, timeout_s=60.0):
    return wait_until(lambda: rd(mmio, REG_STATUS) & (1 << 2), timeout_s, label="compute done_sticky")


## Hex / DDR Buffer Helpers

`golden_gen` 目前的 `.hex` 可能一行是 32-bit，也可能一行是 128-bit row。下面 parser 會把長 hex line 從右到左切成 32-bit word，對應 RTL little-endian byte lane。


In [5]:
def split_hex_token_to_u32_words(token):
    token = token.strip().replace("_", "")
    if not token:
        return []
    if token.startswith("0x") or token.startswith("0X"):
        token = token[2:]
    if len(token) <= 8:
        return [int(token, 16)]
    pad = (-len(token)) % 8
    token = "0" * pad + token
    words = []
    for end in range(len(token), 0, -8):
        words.append(int(token[end-8:end], 16))
    return words


def read_hex_words(path):
    path = Path(path)
    words = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.split("#", 1)[0].split("//", 1)[0].strip()
            if not line:
                continue
            for token in line.replace(",", " ").split():
                words.extend(split_hex_token_to_u32_words(token))
    return np.array(words, dtype=np.uint32)


def pack_u8_to_u32(data):
    arr = np.asarray(data, dtype=np.uint8).reshape(-1)
    n = (len(arr) + 3) // 4
    out = np.zeros(n, dtype=np.uint32)
    for i, b in enumerate(arr):
        out[i // 4] |= np.uint32(int(b) << (8 * (i % 4)))
    return out


def pack_i16_to_u32(data):
    arr = np.asarray(data, dtype=np.int16).reshape(-1).view(np.uint16)
    n = (len(arr) + 1) // 2
    out = np.zeros(n, dtype=np.uint32)
    for i, h in enumerate(arr):
        out[i // 2] |= np.uint32(int(h) << (16 * (i % 2)))
    return out


def alloc_u32(words, cacheable=False):
    if allocate is None:
        raise RuntimeError("pynq.allocate is not available")
    words = np.asarray(words, dtype=np.uint32).reshape(-1)
    buf = allocate(shape=(len(words),), dtype=np.uint32, cacheable=cacheable)
    buf[:] = words
    if hasattr(buf, "flush"):
        buf.flush()
    return buf


## DDR Load / Store Commands


In [6]:
def _ddr_error_from_status(status):
    d = decode_ddr_status(status)
    if d["cmd_error"] or d["load_error"] or d["load_error_state"] or d["store_error"] or d["store_error_state"]:
        return d
    return None


def wait_ddr_idle(mmio, timeout_s=10.0):
    def idle_or_error():
        s = rd(mmio, REG_DDR_STATUS)
        err = _ddr_error_from_status(s)
        if err:
            raise RuntimeError(f"DDR error while waiting idle: status=0x{s:08x}, decoded={err}")
        d = decode_ddr_status(s)
        return not (d["load_busy"] or d["store_busy"])
    return wait_until(idle_or_error, timeout_s=timeout_s, label="DDR idle")


def wait_input_idle(mmio, timeout_s=10.0):
    def idle_or_error():
        s = rd(mmio, REG_STATUS)
        d = decode_status(s)
        if d["input_error"]:
            raise RuntimeError(f"Input loader error while waiting idle: status=0x{s:08x}, decoded={d}")
        return not d["input_busy"]
    return wait_until(idle_or_error, timeout_s=timeout_s, label="input loader idle")




def ddr_debug_snapshot(mmio):
    status = rd(mmio, REG_DDR_STATUS)
    return {
        "ddr_status_hex": f"0x{status:08x}",
        "ddr_status": decode_ddr_status(status),
        "ddr_index": rd(mmio, REG_DDR_INDEX),
        "ddr_src_addr": f"0x{rd(mmio, REG_DDR_SRC_ADDR):08x}",
        "ddr_dst_addr": f"0x{rd(mmio, REG_DDR_DST_ADDR):08x}",
        "ddr_local_base": rd(mmio, REG_DDR_BASE),
        "ddr_word_count": rd(mmio, REG_DDR_COUNT),
        "ddr_target": rd(mmio, REG_DDR_TARGET),
        "input_target": rd(mmio, REG_INPUT_TGT),
        "input_count": rd(mmio, REG_INPUT_COUNT),
        "core_status_hex": f"0x{rd(mmio, REG_STATUS):08x}",
        "core_status": decode_status(rd(mmio, REG_STATUS)),
        "perf_ddr_load_cycles": rd(mmio, REG_PERF_DDR_LD),
        "perf_ddr_read_words": rd(mmio, REG_PERF_DDR_RDW),
        "perf_loader_data_words": rd(mmio, REG_PERF_LDR_WR),
    }


def print_ddr_debug(mmio):
    snap = ddr_debug_snapshot(mmio)
    for k, v in snap.items():
        print(f"{k}: {v}")
    return snap



def host_load_words(mmio, target, words, local_base=0, already_started=False, start_index=0, timeout_s=30.0):
    """Load words through the AXI-Lite input window.

    This is a bring-up fallback for the case where the RTL input loader has
    accepted a DDR load command but the AXI master does not start issuing reads.
    It is slower and counted separately by host_loader_words.
    """
    arr = np.asarray(words, dtype=np.uint32).reshape(-1)
    word_count = int(len(arr))
    start_index = int(start_index)
    if word_count <= 0:
        return None
    if start_index < 0:
        start_index = 0
    if start_index > word_count:
        start_index = word_count

    if not already_started:
        wait_input_idle(mmio, timeout_s=timeout_s)
        wr(mmio, INPUT_BASE_REG, int(local_base))
        wr(mmio, INPUT_COUNT_REG, word_count)
        wr(mmio, INPUT_CTRL, (int(target) << 4) | 1)

    for word in arr[start_index:]:
        wr(mmio, INPUT_DATA_REG, int(word))
    wait_input_idle(mmio, timeout_s=timeout_s)
    return None

def ddr_load_addr(mmio, target, src_addr, word_count, local_base=0, timeout_s=30.0, fallback_words=None):
    """Load a DDR buffer into an RTL-side target through the RTL AXI master.

    Some stage/checkpoint runs clear performance counters while the RTL loader
    still reports completion through INPUT_COUNT/INPUT_DONE.  Therefore load
    completion must not rely only on PERF_DDR_RDW/PERF_LDR_WR deltas.
    """
    word_count = int(word_count)
    target = int(target)
    src_addr = int(src_addr)
    local_base = int(local_base)
    if word_count <= 0:
        return None

    wait_ddr_idle(mmio, timeout_s=timeout_s)
    wait_input_idle(mmio, timeout_s=timeout_s)
    clear_ddr_flags(mmio)

    read_words_before = rd(mmio, REG_PERF_DDR_RDW)
    loader_words_before = rd(mmio, REG_PERF_LDR_WR)

    wr(mmio, REG_DDR_SRC_ADDR, src_addr)
    wr(mmio, REG_DDR_BASE, local_base)
    wr(mmio, REG_DDR_COUNT, word_count)
    wr(mmio, REG_DDR_TARGET, target)
    pulse(mmio, REG_DDR_CONTROL, 1)  # start_load

    def load_progress_words():
        input_target = int(rd(mmio, REG_INPUT_TGT))
        input_count = int(rd(mmio, REG_INPUT_COUNT))
        if input_target == target and input_count >= 0:
            # REG_INPUT_COUNT is zero-based: 95 means 96 words accepted.
            return input_count + 1
        return 0

    def done_or_error():
        s = rd(mmio, REG_DDR_STATUS)
        d = decode_ddr_status(s)
        if d["cmd_error"] or d["load_error"] or d["load_error_state"]:
            raise RuntimeError(f"DDR load failed: status=0x{s:08x}, decoded={d}, debug={ddr_debug_snapshot(mmio)}")

        read_delta = (rd(mmio, REG_PERF_DDR_RDW) - read_words_before) & 0xFFFFFFFF
        loader_delta = (rd(mmio, REG_PERF_LDR_WR) - loader_words_before) & 0xFFFFFFFF
        input_progress = load_progress_words()

        done_by_counter = int(read_delta) >= word_count or int(loader_delta) >= word_count
        done_by_input = int(input_progress) >= word_count
        return d["load_done"] and (not d["load_busy"]) and (done_by_counter or done_by_input)

    def fallback_if_ddr_idle(reason):
        if (not ENABLE_AXI_LITE_FALLBACK) or fallback_words is None:
            return False
        snap = ddr_debug_snapshot(mmio)
        read_delta_now = (rd(mmio, REG_PERF_DDR_RDW) - read_words_before) & 0xFFFFFFFF
        loader_delta_now = (rd(mmio, REG_PERF_LDR_WR) - loader_words_before) & 0xFFFFFFFF
        input_count_now = int(snap.get("input_count", 0))
        core_status_now = snap.get("core_status", {})
        ddr_status_now = snap.get("ddr_status", {})
        if (read_delta_now == 0 and loader_delta_now == 0 and
                core_status_now.get("input_busy", False) and
                int(snap.get("input_target", -1)) == target and
                not ddr_status_now.get("load_busy", False)):
            print(
                f"WARNING: DDR load did not start for target={target}, base={local_base} ({reason}). "
                f"Falling back to AXI-Lite DATA writes from word {input_count_now}/{word_count}."
            )
            host_load_words(
                mmio,
                target,
                fallback_words,
                local_base=local_base,
                already_started=True,
                start_index=input_count_now,
                timeout_s=timeout_s,
            )
            return True
        return False

    if fallback_words is not None:
        probe_t0 = time.time()
        while time.time() - probe_t0 < min(0.20, timeout_s):
            if done_or_error():
                wait_input_idle(mmio, timeout_s=timeout_s)
                return None
            read_delta_probe = (rd(mmio, REG_PERF_DDR_RDW) - read_words_before) & 0xFFFFFFFF
            loader_delta_probe = (rd(mmio, REG_PERF_LDR_WR) - loader_words_before) & 0xFFFFFFFF
            ddr_busy_probe = decode_ddr_status(rd(mmio, REG_DDR_STATUS))["load_busy"]
            if read_delta_probe > 0 or loader_delta_probe > 0 or ddr_busy_probe or load_progress_words() > 0:
                break
            if fallback_if_ddr_idle("early-stall"):
                return None
            time.sleep(0.001)

    try:
        wait_until(done_or_error, timeout_s=timeout_s, label="DDR load done")
        wait_input_idle(mmio, timeout_s=timeout_s)
    except TimeoutError as e:
        if fallback_if_ddr_idle("timeout"):
            return None
        snap = ddr_debug_snapshot(mmio)
        read_delta_now = (rd(mmio, REG_PERF_DDR_RDW) - read_words_before) & 0xFFFFFFFF
        loader_delta_now = (rd(mmio, REG_PERF_LDR_WR) - loader_words_before) & 0xFFFFFFFF
        input_progress_now = load_progress_words()
        raise TimeoutError(
            f"DDR load timeout: target={target}, src=0x{src_addr:08x}, words={word_count}, "
            f"local_base={local_base}, read_delta={read_delta_now}, loader_delta={loader_delta_now}, "
            f"input_progress={input_progress_now}, debug={snap}"
        ) from e
    return None



def ddr_load(mmio, target, words_or_buf, local_base=0, timeout_s=30.0):
    """Copy a numpy/PYNQ buffer or Python word list into an RTL local target via AXI master DDR load."""
    if allocate is None:
        raise RuntimeError("pynq.allocate is not available")
    fallback_words = None
    if hasattr(words_or_buf, "physical_address"):
        buf = words_or_buf
    else:
        arr = np.asarray(words_or_buf, dtype=np.uint32)
        fallback_words = arr
        buf = allocate(shape=(len(arr),), dtype=np.uint32, cacheable=False)
        if len(arr):
            buf[:] = arr
    if hasattr(buf, "flush"):
        buf.flush()
    ddr_load_addr(
        mmio,
        target,
        int(buf.physical_address),
        len(buf),
        local_base=local_base,
        timeout_s=timeout_s,
        fallback_words=fallback_words,
    )
    return buf

def ddr_store_addr(mmio, target, dst_addr, word_count, local_base=0, timeout_s=60.0):
    """Store an RTL-side buffer to DDR through the RTL AXI master.

    Stage dump stores may complete in RTL without incrementing the generic
    PERF_DDR_WRW delta counter. Therefore completion must accept the hardware
    store_done/store_idle state plus ddr_index reaching the last requested word.
    """
    before_words = rd(mmio, REG_PERF_DDR_WRW) if "REG_PERF_DDR_WRW" in globals() else 0

    wr(mmio, REG_DDR_DST_ADDR, int(dst_addr))
    wr(mmio, REG_DDR_BASE, int(local_base))
    wr(mmio, REG_DDR_COUNT, int(word_count))
    wr(mmio, REG_DDR_TARGET, int(target))
    pulse(mmio, REG_DDR_CONTROL, 2)  # start_store

    def done_or_error():
        s = rd(mmio, REG_DDR_STATUS)
        d = decode_ddr_status(s)
        if d["cmd_error"] or d["store_error"] or d["store_error_state"]:
            raise RuntimeError(f"DDR store failed: status=0x{s:08x}, decoded={d}, debug={ddr_debug_snapshot(mmio)}")

        idx = rd(mmio, REG_DDR_INDEX)
        after_words = rd(mmio, REG_PERF_DDR_WRW) if "REG_PERF_DDR_WRW" in globals() else before_words
        write_delta = (after_words - before_words) & 0xFFFFFFFF

        done_by_index = int(word_count) == 0 or int(idx) >= int(word_count) - 1
        done_by_counter = int(write_delta) >= int(word_count)
        return d["store_done"] and (not d["store_busy"]) and (done_by_index or done_by_counter)

    try:
        wait_until(done_or_error, timeout_s=timeout_s, label="DDR store done")
    except TimeoutError as e:
        snap = ddr_debug_snapshot(mmio)
        after_words = rd(mmio, REG_PERF_DDR_WRW) if "REG_PERF_DDR_WRW" in globals() else before_words
        write_delta_now = (after_words - before_words) & 0xFFFFFFFF
        raise TimeoutError(
            f"DDR store timeout: target={target}, dst=0x{int(dst_addr):08x}, words={word_count}, "
            f"local_base={local_base}, write_delta={write_delta_now}, debug={snap}"
        ) from e
    return None


def ddr_store_to_buffer(mmio, target, out, word_count, local_base=0, dst_word_offset=0, timeout_s=30.0):
    if hasattr(out, "flush"):
        out.flush()
    dst_addr = int(out.physical_address) + int(dst_word_offset) * 4
    ddr_store_addr(mmio, target, dst_addr, int(word_count), local_base=local_base, timeout_s=timeout_s)
    if hasattr(out, "invalidate"):
        out.invalidate()
    return out


def ddr_store_xout(mmio, word_count, local_base=0, timeout_s=30.0, cacheable=False):
    if allocate is None:
        raise RuntimeError("pynq.allocate is not available")
    out = allocate(shape=(int(word_count),), dtype=np.uint32, cacheable=cacheable)
    out[:] = 0
    return ddr_store_to_buffer(mmio, target=0, out=out, word_count=word_count, local_base=local_base, timeout_s=timeout_s)


## Load Golden / Smoke Data

目前 `golden_gen/hex/case_vit_qat` 比較像單層矩陣運算輸出的資料包，不一定包含完整 raw image、position、CLS、gamma、每一層所有 transformer tile。下面提供 smoke data loader：有檔案就讀，缺的就用安全預設值補，先確認 AXI/DDR/RTL flow 能跑。

真正 full-image accuracy test 需要 golden generator 匯出完整 ViT 所需的 memory layout：image、pos、cls、gamma、patch weight/bias tiles、transformer QKV/OutProj/FC1/FC2 weight/bias tiles。


In [7]:
IMG_H = 224
IMG_W = 224
IMG_C = 3
PATCH_SIZE = 16
EMBED_DIM = 384
FFN_CHANNEL_NUM = 1536
PATCH_COUNT = (IMG_H // PATCH_SIZE) * (IMG_W // PATCH_SIZE)
TOKEN_NUM = PATCH_COUNT + 1
TOKEN_TILE = 8
CHANNEL_TILE = 8
TILE_ROW_WORDS = CHANNEL_TILE // 4
WEIGHT_TILE_WORDS = CHANNEL_TILE * TILE_ROW_WORDS
BIAS_TILE_WORDS = CHANNEL_TILE
PL_CLK_HZ = 25_000_000
RTL_SYSTOLIC_PEAK_MAC_PER_CYCLE = 64
THEORY_PROFILING_PEAK_MAC_PER_CYCLE = 128
PAD_TOKEN_NUM = ((TOKEN_NUM + TOKEN_TILE - 1) // TOKEN_TILE) * TOKEN_TILE
XOUT_ELEMS = TOKEN_NUM * EMBED_DIM
X_WORDS = (XOUT_ELEMS + 3) // 4

IMAGE_BYTES = IMG_H * IMG_W * IMG_C
POS_BYTES = TOKEN_NUM * EMBED_DIM
CLS_BYTES = EMBED_DIM
GAMMA_HALVES = 2 * EMBED_DIM
PAGE_WORDS = 1024
GELU_WORDS = (PAD_TOKEN_NUM * FFN_CHANNEL_NUM) // 4

IMAGE_WORDS = (IMAGE_BYTES + 3) // 4
POS_WORDS = (POS_BYTES + 3) // 4
CLS_WORDS = (CLS_BYTES + 3) // 4


def find_case_file(name):
    p = GOLDEN_ROOT / name
    return p if p.exists() else None


def find_first_case_file(names):
    for name in names:
        p = find_case_file(name)
        if p:
            return p
    return None


def load_case_manifest():
    p = find_case_file("manifest.json")
    if not p:
        return {}
    return json.loads(p.read_text(encoding="utf-8"))


def manifest_patch_shift(default=0):
    m = load_case_manifest()
    return int(m.get("patch_requant_shift", default))


def pad_or_trim_words(words, count):
    words = np.asarray(words, dtype=np.uint32).reshape(-1)
    count = int(count)
    if len(words) == count:
        return words
    out = np.zeros(count, dtype=np.uint32)
    out[:min(len(words), count)] = words[:count]
    return out


def normalize_u8_words(raw_words, byte_count):
    raw = np.asarray(raw_words, dtype=np.uint32).reshape(-1)
    word_count = (int(byte_count) + 3) // 4
    if len(raw) == word_count:
        return raw
    if len(raw) >= byte_count and np.all((raw[:byte_count] & np.uint32(0xFFFFFF00)) == 0):
        return pack_u8_to_u32(raw[:byte_count].astype(np.uint8))
    return pad_or_trim_words(raw, word_count)


def default_image_words(value=128):
    return pack_u8_to_u32(np.full(IMAGE_BYTES, value, dtype=np.uint8))


def default_pos_words(value=128):
    return pack_u8_to_u32(np.full(POS_BYTES, value, dtype=np.uint8))


def default_cls_words(value=128):
    return pack_u8_to_u32(np.full(CLS_BYTES, value, dtype=np.uint8))


def default_gamma_words(value=16384):
    # RTL target 4 writes one gamma per 32-bit DATA beat.
    # DATA[15:0] is the signed int16 gamma value; high 16 bits are ignored.
    return np.full(GAMMA_HALVES, np.uint32(value & 0xFFFF), dtype=np.uint32)


def gamma_words_for_loader(raw_words):
    raw = np.asarray(raw_words, dtype=np.uint32).reshape(-1)
    if len(raw) == GAMMA_HALVES:
        return raw & np.uint32(0xFFFF)
    if len(raw) == EMBED_DIM:
        one = raw & np.uint32(0xFFFF)
        return np.concatenate([one, one]).astype(np.uint32)

    # Backward compatibility: older files may pack two int16 gamma values per 32-bit word.
    vals = []
    for w in raw:
        vals.append(int(w) & 0xFFFF)
        vals.append((int(w) >> 16) & 0xFFFF)
    if len(vals) == EMBED_DIM:
        vals = vals + vals
    if len(vals) < GAMMA_HALVES:
        vals.extend([0x4000] * (GAMMA_HALVES - len(vals)))
    return np.asarray(vals[:GAMMA_HALVES], dtype=np.uint32)


def load_static_inputs(mmio, cacheable=False):
    image_p = find_first_case_file(["image.hex", "raw_image.hex", "act_in.hex"])
    pos_p = find_first_case_file(["pos.hex", "position.hex", "pos_embed.hex"])
    cls_p = find_case_file("cls.hex")
    gamma_p = find_first_case_file(["gamma.hex", "gamma.mem"])

    image_words = normalize_u8_words(read_hex_words(image_p), IMAGE_BYTES) if image_p else default_image_words()
    pos_words = normalize_u8_words(read_hex_words(pos_p), POS_BYTES) if pos_p else default_pos_words()
    cls_words = normalize_u8_words(read_hex_words(cls_p), CLS_BYTES) if cls_p else default_cls_words()
    gamma_words = gamma_words_for_loader(read_hex_words(gamma_p)) if gamma_p else default_gamma_words()

    # image/pos are intentionally not loaded as full on-chip memories anymore.
    # They stay in PS DDR and are copied into 1K-word RTL page caches only when PAGE_REQ asks for them.
    image_buf = alloc_u32(pad_or_trim_words(image_words, IMAGE_WORDS), cacheable=cacheable)
    pos_buf = alloc_u32(pad_or_trim_words(pos_words, POS_WORDS), cacheable=cacheable)

    print(f"Image DDR buffer: {len(image_buf)} words @ 0x{int(image_buf.physical_address):08x}")
    print(f"Position DDR buffer: {len(pos_buf)} words @ 0x{int(pos_buf.physical_address):08x}")
    print("Loading cls words into RTL:", len(cls_words))
    cls_buf = ddr_load(mmio, TARGET_CLS, pad_or_trim_words(cls_words, CLS_WORDS), local_base=0)
    print("Loading gamma words into RTL:", len(gamma_words))
    gamma_buf = ddr_load(mmio, TARGET_GAMMA, gamma_words, local_base=0)

    return {
        "image": image_buf,
        "pos": pos_buf,
        "cls": cls_buf,
        "gamma": gamma_buf,
        "image_file": image_p,
        "pos_file": pos_p,
        "cls_file": cls_p,
        "gamma_file": gamma_p,
    }


def load_tile_from_words(mmio, target, all_words, tile_id, words_per_tile):
    start = int(tile_id) * int(words_per_tile)
    stop = start + int(words_per_tile)

    if hasattr(all_words, "physical_address"):
        total = len(all_words)
        if start >= total:
            tile = np.zeros(words_per_tile, dtype=np.uint32)
            return ddr_load(mmio, target, tile, local_base=int(tile_id))
        count = min(int(words_per_tile), total - start)
        if count == int(words_per_tile):
            if hasattr(all_words, "flush"):
                all_words.flush()
            src_addr = int(all_words.physical_address) + start * 4
            fallback_tile = np.asarray(all_words[start:start+count], dtype=np.uint32).copy()
            ddr_load_addr(mmio, target, src_addr, count, local_base=int(tile_id), fallback_words=fallback_tile)
            return all_words
        tile = np.zeros(words_per_tile, dtype=np.uint32)
        tile[:count] = np.asarray(all_words[start:start+count], dtype=np.uint32)
        return ddr_load(mmio, target, tile, local_base=int(tile_id))

    tile = np.asarray(all_words[start:stop], dtype=np.uint32)
    if len(tile) < words_per_tile:
        padded = np.zeros(words_per_tile, dtype=np.uint32)
        padded[:len(tile)] = tile
        tile = padded
    return ddr_load(mmio, target, tile, local_base=int(tile_id))


def load_requested_tiles_once(mmio, patch_weight_words=None, patch_bias_words=None, trans_weight_words=None, trans_bias_words=None, verbose=False):
    """Service only the tile types that RTL is actively requesting.

    Important: callers may pass all source buffers, but this function must still
    look at the hardware miss bits. Otherwise Python can issue a DDR load for a
    tile that is already matched; the RTL loader will ignore it and Python waits
    forever for a new done edge.
    """
    patch_req = rd(mmio, REG_PATCH_REQ)
    trans_req = rd(mmio, REG_TRANS_REQ)
    patch_w_tile = patch_req & 0xFFFF
    patch_b_tile = (patch_req >> 16) & 0xFFFF
    trans_w_tile = trans_req & 0xFFFF
    trans_b_tile = (trans_req >> 16) & 0xFFFF
    status_now = decode_status(rd(mmio, REG_STATUS))
    actions = []

    if patch_weight_words is not None and status_now.get("patch_weight_miss", False):
        actions.append(f"PW{patch_w_tile}")
        load_tile_from_words(mmio, TARGET_PATCH_W, patch_weight_words, patch_w_tile, WEIGHT_TILE_WORDS)

    if patch_bias_words is not None and status_now.get("patch_bias_miss", False):
        actions.append(f"PB{patch_b_tile}")
        load_tile_from_words(mmio, TARGET_PATCH_B, patch_bias_words, patch_b_tile, BIAS_TILE_WORDS)

    if trans_weight_words is not None and status_now.get("trans_weight_miss", False):
        actions.append(f"TW{trans_w_tile}")
        load_tile_from_words(mmio, TARGET_TRANS_W, trans_weight_words, trans_w_tile, WEIGHT_TILE_WORDS)

    if trans_bias_words is not None and status_now.get("trans_bias_miss", False):
        actions.append(f"TB{trans_b_tile}")
        load_tile_from_words(mmio, TARGET_TRANS_B, trans_bias_words, trans_b_tile, BIAS_TILE_WORDS)

    if verbose and actions:
        print(
            "TILE service "
            f"actions={actions} "
            f"patch_req=0x{patch_req:08x} (W={patch_w_tile}, B={patch_b_tile}) "
            f"trans_req=0x{trans_req:08x} (W={trans_w_tile}, B={trans_b_tile}) "
            f"status={status_now}"
        )

    return patch_req, trans_req


def load_full_design_words():
    data = {}
    candidates = {
        "patch_weight": ["patch_weight.hex", "pe_weight.hex", "weight.hex"],
        "patch_bias": ["patch_bias.hex", "pe_bias.hex", "bias.hex"],
        "trans_weight": ["transformer_weight.hex", "trans_weight.hex", "weight.hex"],
        "trans_bias": ["transformer_bias.hex", "trans_bias.hex", "bias.hex"],
        "x_out": ["x_out.hex", "expected_x_out.hex", "act_out.hex"],
    }
    for key, names in candidates.items():
        for name in names:
            p = find_case_file(name)
            if p:
                data[key] = read_hex_words(p)
                print(f"{key}: {len(data[key])} words from {p}")
                break
        if key not in data:
            print(f"{key}: missing from {GOLDEN_ROOT}")
    return data


def alloc_weight_buffers(case_words, cacheable=False):
    buffers = {}
    for key in ["patch_weight", "patch_bias", "trans_weight", "trans_bias"]:
        if key in case_words and case_words[key] is not None:
            buffers[key] = alloc_u32(case_words[key], cacheable=cacheable)
            print(f"{key} DDR buffer: {len(buffers[key])} words @ 0x{int(buffers[key].physical_address):08x}")
        else:
            buffers[key] = None
    return buffers


def golden_xout_u8(golden_words=None):
    if golden_words is None:
        p = find_first_case_file(["x_out.hex", "expected_x_out.hex", "act_out.hex"])
        if not p:
            raise FileNotFoundError(f"No x_out golden file under {GOLDEN_ROOT}")
        golden_words = read_hex_words(p)

    words = np.asarray(golden_words, dtype=np.uint32).reshape(-1)
    if len(words) >= XOUT_ELEMS:
        return (words[:XOUT_ELEMS] & np.uint32(0xFF)).astype(np.uint8)

    unpacked = np.zeros(len(words) * 4, dtype=np.uint8)
    for i, w in enumerate(words):
        unpacked[i*4 + 0] = int(w) & 0xFF
        unpacked[i*4 + 1] = (int(w) >> 8) & 0xFF
        unpacked[i*4 + 2] = (int(w) >> 16) & 0xFF
        unpacked[i*4 + 3] = (int(w) >> 24) & 0xFF
    return unpacked[:XOUT_ELEMS]


def compare_xout_arrays(rtl_u8, golden_u8, tolerance=0, tolerances=(0, 1, 2, 4)):
    rtl = np.asarray(rtl_u8, dtype=np.uint8).reshape(-1)[:XOUT_ELEMS]
    golden = np.asarray(golden_u8, dtype=np.uint8).reshape(-1)[:XOUT_ELEMS]
    if len(rtl) != len(golden):
        raise ValueError(f"Length mismatch: rtl={len(rtl)} golden={len(golden)}")

    diff = rtl.astype(np.int16) - golden.astype(np.int16)
    abs_diff = np.abs(diff)
    total = int(len(abs_diff))
    bad = np.nonzero(abs_diff > int(tolerance))[0]
    exact = int(np.count_nonzero(abs_diff == 0))
    within = {
        f"within_{int(t)}_lsb_percent": (100.0 * float(np.count_nonzero(abs_diff <= int(t))) / total) if total else 100.0
        for t in tolerances
    }
    return {
        "match": len(bad) == 0,
        "total_values": total,
        "mismatch_count": int(len(bad)),
        "mismatch_percent": (100.0 * float(len(bad)) / total) if total else 0.0,
        "exact_match_percent": (100.0 * float(exact) / total) if total else 100.0,
        "mean_abs_diff_lsb": float(abs_diff.mean()) if total else 0.0,
        "mean_abs_diff_percent_of_255": (100.0 * float(abs_diff.mean()) / 255.0) if total else 0.0,
        "max_abs_diff": int(abs_diff.max()) if total else 0,
        "rmse_lsb": float(np.sqrt(np.mean(diff.astype(np.float64) ** 2))) if total else 0.0,
        "mean_signed_diff_lsb": float(diff.mean()) if total else 0.0,
        "tolerance": int(tolerance),
        **within,
    }, diff, abs_diff, bad


def verify_xout(out_buf, golden_words=None, tolerance=0, max_print=20):
    # RTL X_out store path writes one uint8 activation in the low byte of each 32-bit word.
    rtl = (np.asarray(out_buf, dtype=np.uint32).reshape(-1)[:XOUT_ELEMS] & np.uint32(0xFF)).astype(np.uint8)
    golden = golden_xout_u8(golden_words)
    result, diff, abs_diff, bad = compare_xout_arrays(rtl, golden, tolerance=tolerance)

    if globals().get("PRINT_ACCURACY_METRICS", False):
        print("X_out verify:")
        print(f"  exact match      : {result['exact_match_percent']:.4f}%")
        print(f"  mismatch         : {result['mismatch_percent']:.4f}%  ({result['mismatch_count']} / {result['total_values']})")
        print(f"  mean abs diff    : {result['mean_abs_diff_lsb']:.4f} LSB ({result['mean_abs_diff_percent_of_255']:.4f}% of 255)")
        print(f"  max abs diff     : {result['max_abs_diff']} LSB")
        print(f"  within 1/2/4 LSB : {result['within_1_lsb_percent']:.4f}% / {result['within_2_lsb_percent']:.4f}% / {result['within_4_lsb_percent']:.4f}%")
        for idx in bad[:max_print]:
            print(f"  idx={idx}: rtl={int(rtl[idx])} golden={int(golden[idx])} diff={int(diff[idx])}")
    return result


## Run Flow Skeleton

Full design 需要依照 `PATCH_REQ` / `TRANS_REQ` 持續補 tile。下面是 bring-up skeleton；真正 full accuracy 要先把 golden generator 匯出的 tile layout 對上 RTL address layout。


In [8]:
def alloc_gelu_buffer(cacheable=False):
    if allocate is None:
        raise RuntimeError("pynq.allocate is not available")
    buf = allocate(shape=(GELU_WORDS,), dtype=np.uint32, cacheable=cacheable)
    buf[:] = 0
    if hasattr(buf, "flush"):
        buf.flush()
    print(f"GELU_out DDR buffer: {len(buf)} words @ 0x{int(buf.physical_address):08x}")
    return buf


def _page_count_for_buffer(buf, base_word):
    base_word = int(base_word)
    total = len(buf)
    if base_word >= total:
        return 0
    return min(PAGE_WORDS, total - base_word)


def _wait_page_req_change(mmio, old_req_word, timeout_s=10.0):
    t0 = time.time()
    while True:
        now = rd(mmio, REG_PAGE_REQ)
        dec = decode_page_req(now)
        if (not dec["valid"]) or now != old_req_word:
            return now
        if time.time() - t0 > timeout_s:
            raise TimeoutError(
                f"PAGE_REQ did not clear/change after service: PAGE_REQ=0x{old_req_word:08x}, "
                f"page_decoded={decode_page_req(old_req_word)}, ddr_debug={ddr_debug_snapshot(mmio)}"
            )
        time.sleep(0.0002)


def service_page_request_once(mmio, page_buffers, timeout_s=30.0, verbose=False):
    req_word = rd(mmio, REG_PAGE_REQ)
    req = decode_page_req(req_word)
    if not req["valid"]:
        return False

    target = req["target"]
    base = int(req["base"])
    is_store = bool(req["store"])

    if target == TARGET_IMAGE:
        if is_store:
            raise RuntimeError(f"Unexpected image store PAGE_REQ: 0x{req_word:08x}")
        buf = page_buffers["image"]
        count = _page_count_for_buffer(buf, base)
        if count <= 0:
            raise RuntimeError(f"Image PAGE_REQ base out of range: base={base}, words={len(buf)}")
        if hasattr(buf, "flush"):
            buf.flush()
        if verbose:
            print(f"PAGE load image base={base} count={count}")
        ddr_load_addr(mmio, TARGET_IMAGE, int(buf.physical_address) + base * 4, count, local_base=base, timeout_s=timeout_s, fallback_words=np.asarray(buf, dtype=np.uint32).reshape(-1)[base:base + count])

    elif target == TARGET_POS:
        if is_store:
            raise RuntimeError(f"Unexpected pos store PAGE_REQ: 0x{req_word:08x}")
        buf = page_buffers["pos"]
        count = _page_count_for_buffer(buf, base)
        if count <= 0:
            raise RuntimeError(f"Pos PAGE_REQ base out of range: base={base}, words={len(buf)}")
        if hasattr(buf, "flush"):
            buf.flush()
        if verbose:
            print(f"PAGE load pos base={base} count={count}")
        ddr_load_addr(mmio, TARGET_POS, int(buf.physical_address) + base * 4, count, local_base=base, timeout_s=timeout_s, fallback_words=np.asarray(buf, dtype=np.uint32).reshape(-1)[base:base + count])

    elif target == TARGET_GELU_PAGE:
        buf = page_buffers["gelu"]
        count = _page_count_for_buffer(buf, base)
        if count <= 0:
            raise RuntimeError(f"GELU PAGE_REQ base out of range: base={base}, words={len(buf)}")
        addr = int(buf.physical_address) + base * 4
        if is_store:
            if verbose:
                print(f"PAGE store GELU base={base} count={count}")
            ddr_store_addr(mmio, TARGET_GELU_PAGE, addr, count, local_base=base, timeout_s=timeout_s)
            if hasattr(buf, "invalidate"):
                buf.invalidate()
        else:
            if hasattr(buf, "flush"):
                buf.flush()
            if verbose:
                print(f"PAGE load GELU base={base} count={count}")
            ddr_load_addr(mmio, TARGET_GELU_PAGE, addr, count, local_base=base, timeout_s=timeout_s, fallback_words=np.asarray(buf, dtype=np.uint32).reshape(-1)[base:base + count])
    else:
        raise RuntimeError(f"Unknown PAGE_REQ target={target}, raw=0x{req_word:08x}")

    _wait_page_req_change(mmio, req_word)
    return True



def _decode_req_pair(req_word):
    return int(req_word) & 0xFFFF, (int(req_word) >> 16) & 0xFFFF


def print_progress_snapshot(mmio, elapsed_s, status_decoded=None, patch_req=None, trans_req=None):
    if status_decoded is None:
        status_decoded = decode_status(rd(mmio, REG_STATUS))
    if patch_req is None:
        patch_req = rd(mmio, REG_PATCH_REQ)
    if trans_req is None:
        trans_req = rd(mmio, REG_TRANS_REQ)

    patch_w, patch_b = _decode_req_pair(patch_req)
    trans_w, trans_b = _decode_req_pair(trans_req)
    ddr_status_word = rd(mmio, REG_DDR_STATUS)
    ddr_status = decode_ddr_status(ddr_status_word)
    ddr_index = rd(mmio, REG_DDR_INDEX)
    perf_total = rd(mmio, REG_PERF_TOTAL)
    perf_ddr_ld = rd(mmio, REG_PERF_DDR_LD)
    perf_ddr_rdw = rd(mmio, REG_PERF_DDR_RDW)
    perf_ldr = rd(mmio, REG_PERF_LDR_WR)

    if status_decoded.get("patch_external_phase"):
        phase = "PATCH"
        req_text = f"PW={patch_w} PB={patch_b}"
    elif status_decoded.get("trans_external_phase"):
        phase = "TRANS"
        req_text = f"TW={trans_w} TB={trans_b}"
    elif ddr_status.get("load_busy"):
        phase = "DDR_LOAD"
        req_text = f"idx={ddr_index}"
    elif ddr_status.get("store_busy"):
        phase = "DDR_STORE"
        req_text = f"idx={ddr_index}"
    else:
        phase = "COMPUTE"
        req_text = f"PW={patch_w} TW={trans_w}"

    miss = []
    for key, label in [
        ("patch_weight_miss", "PW"),
        ("patch_bias_miss", "PB"),
        ("trans_weight_miss", "TW"),
        ("trans_bias_miss", "TB"),
    ]:
        if status_decoded.get(key):
            miss.append(label)
    miss_text = ",".join(miss) if miss else "-"

    print(
        f"[progress {elapsed_s:8.1f}s] "
        f"phase={phase:<9} req={req_text:<18} miss={miss_text:<5} "
        f"ddr={ddr_status.get('load_state_name','?')}/{ddr_status.get('store_state_name','?')} "
        f"idx={ddr_index} rdw={perf_ddr_rdw} ldr={perf_ldr} "
        f"ldcyc={perf_ddr_ld} total={perf_total}",
        flush=True,
    )

def service_tiles_until_done(mmio, patch_weight_words=None, patch_bias_words=None, trans_weight_words=None, trans_bias_words=None,
                             page_buffers=None, timeout_s=900.0, verbose_pages=False, progress_interval_s=5.0):
    t0 = time.time()
    last_progress_t = t0
    last_patch_req = None
    last_trans_req = None
    last_tile_log_t = 0.0
    last_progress_print_t = t0 - float(progress_interval_s or 0.0)
    while True:
        if page_buffers is not None and service_page_request_once(mmio, page_buffers, verbose=verbose_pages):
            last_progress_t = time.time()
            continue

        status = rd(mmio, REG_STATUS)
        ds = decode_status(status)
        if ds["done_sticky"]:
            return status
        if ds["input_error"]:
            raise RuntimeError(f"Input loader error: STATUS=0x{status:08x}, decoded={ds}")

        patch_req = rd(mmio, REG_PATCH_REQ)
        trans_req = rd(mmio, REG_TRANS_REQ)
        need_patch = ds["patch_weight_miss"] or ds["patch_bias_miss"]
        need_trans = ds["trans_weight_miss"] or ds["trans_bias_miss"]

        now = time.time()
        if progress_interval_s and (now - last_progress_print_t >= progress_interval_s):
            print_progress_snapshot(mmio, now - t0, ds, patch_req, trans_req)
            last_progress_print_t = now

        log_tiles = verbose_pages and ((patch_req != last_patch_req) or (trans_req != last_trans_req) or (time.time() - last_tile_log_t > 2.0))
        did_tile_service = False
        if need_patch and (patch_weight_words is not None or patch_bias_words is not None):
            load_requested_tiles_once(
                mmio,
                patch_weight_words if ds["patch_weight_miss"] else None,
                patch_bias_words if ds["patch_bias_miss"] else None,
                None,
                None,
                verbose=log_tiles,
            )
            last_patch_req = patch_req
            did_tile_service = True
            if log_tiles:
                last_tile_log_t = time.time()
        if need_trans and (trans_weight_words is not None or trans_bias_words is not None):
            load_requested_tiles_once(
                mmio,
                None,
                None,
                trans_weight_words if ds["trans_weight_miss"] else None,
                trans_bias_words if ds["trans_bias_miss"] else None,
                verbose=log_tiles,
            )
            last_trans_req = trans_req
            did_tile_service = True
            if log_tiles:
                last_tile_log_t = time.time()
        if did_tile_service:
            last_progress_t = time.time()

        if time.time() - last_progress_t > timeout_s:
            page_req = rd(mmio, REG_PAGE_REQ)
            debug_word = rd(mmio, REG_DEBUG)
            raise TimeoutError(
                f"compute timeout after {timeout_s}s without progress: STATUS=0x{status:08x}, decoded={ds}, "
                f"DEBUG={decode_debug_word(debug_word)}, "
                f"PATCH_REQ=0x{patch_req:08x}, TRANS_REQ=0x{trans_req:08x}, "
                f"PAGE_REQ=0x{page_req:08x}, PAGE_DEC={decode_page_req(page_req)}, "
                f"DDR_DEBUG={ddr_debug_snapshot(mmio)}"
            )
        time.sleep(0.0005)


def run_smoke(mmio, patch_shift=None, verify=True, verbose_pages=False, tolerance=0, progress_interval_s=5.0):
    clear_done(mmio)
    clear_perf(mmio)
    case_manifest = load_case_manifest()
    if patch_shift is None:
        patch_shift = int(case_manifest.get("patch_requant_shift", 0))
    print("Golden root:", GOLDEN_ROOT)
    print("Golden case kind:", case_manifest.get("kind", "unknown"))
    print("Using patch_shift:", patch_shift)
    wr(mmio, REG_PATCH_SHIFT, patch_shift)
    wr(mmio, REG_RUN_MODE, 0)

    static_buffers = load_static_inputs(mmio)
    gelu_buf = alloc_gelu_buffer()
    page_buffers = {
        "image": static_buffers["image"],
        "pos": static_buffers["pos"],
        "gelu": gelu_buf,
    }

    case_words = load_full_design_words()
    case_words["manifest"] = case_manifest
    weight_buffers = alloc_weight_buffers(case_words)

    def choose_source(key):
        buf = weight_buffers.get(key)
        return buf if buf is not None else case_words.get(key)

    patch_weight_src = choose_source("patch_weight")
    patch_bias_src = choose_source("patch_bias")
    trans_weight_src = choose_source("trans_weight")
    trans_bias_src = choose_source("trans_bias")

    load_requested_tiles_once(mmio, patch_weight_src, patch_bias_src, trans_weight_src, trans_bias_src)
    start_compute(mmio)
    service_tiles_until_done(
        mmio,
        patch_weight_src,
        patch_bias_src,
        trans_weight_src,
        trans_bias_src,
        page_buffers=page_buffers,
        verbose_pages=verbose_pages,
        progress_interval_s=progress_interval_s,
    )
    out_buf = ddr_store_xout(mmio, XOUT_ELEMS, local_base=0)
    counters = read_perf_counters(mmio)
    verify_result = verify_xout(out_buf, case_words.get("x_out"), tolerance=tolerance) if verify else None

    run_smoke.last_buffers = {
        "static": static_buffers,
        "gelu": gelu_buf,
        "weights": weight_buffers,
    }
    return out_buf, counters, verify_result

# Example:
# overlay, mmio = load_overlay()
# out_buf, counters, verify_result = run_smoke(mmio, verify=True, verbose_pages=True)
# print(counters)
# print(verify_result)



def run_one_block_real_model(mmio, patch_shift=None, verify=True, verbose_pages=False, tolerance=0, progress_interval_s=5.0):
    """Run the current RTL one-transformer-block real-model test and compare X_out."""
    return run_smoke(
        mmio,
        patch_shift=patch_shift,
        verify=verify,
        verbose_pages=verbose_pages,
        tolerance=tolerance,
        progress_interval_s=progress_interval_s,
    )


## RTL Performance Counters and Metrics

這些 counter 是 RTL 每個 clock 真實累加，Python 只負責讀出和換算。


In [9]:
def read_perf_counters(mmio):
    return {name: rd(mmio, off) for name, off in PERF_REGS.items()}


def _counter_u64(counters, lo_name, hi_name):
    return (int(counters.get(hi_name, 0)) << 32) | int(counters.get(lo_name, 0))


def summarize_perf(counters, clk_hz=PL_CLK_HZ):
    total_cycles = int(counters.get("total_cycles", 0))
    compute_cycles = int(counters.get("compute_busy_cycles", 0))
    ddr_load_cycles = int(counters.get("ddr_load_cycles", 0))
    ddr_store_cycles = int(counters.get("ddr_store_cycles", 0))
    ddr_access_cycles = ddr_load_cycles + ddr_store_cycles
    raw_ddr_read_words = int(counters.get("ddr_read_words", 0))
    loader_data_words = int(counters.get("loader_data_words", 0))
    host_loader_words = int(counters.get("host_loader_words", 0))
    dma_loader_words = max(0, loader_data_words - host_loader_words)
    # On some PS/SmartConnect builds, the AXI read-beat counter under-counts;
    # the loader-data counter reflects the actual 32-bit words delivered from DDR.
    ddr_read_words = max(raw_ddr_read_words, dma_loader_words)
    ddr_write_words = int(counters.get("ddr_write_words", 0))
    ddr_read_transactions = int(counters.get("ddr_read_transactions", 0))
    ddr_write_transactions = int(counters.get("ddr_write_transactions", 0))
    ddr_transactions = ddr_read_transactions + ddr_write_transactions
    ddr_words = ddr_read_words + ddr_write_words
    ddr_bytes = 4 * ddr_words

    bram_read_words = int(counters.get("bram_read_words", 0))
    bram_write_words = int(counters.get("bram_write_words", 0))
    bram_words = bram_read_words + bram_write_words
    bram_bytes = 4 * bram_words
    bram_access_cycles = int(counters.get("bram_access_cycles", 0))
    mac_ops = _counter_u64(counters, "mac_ops_lo", "mac_ops_hi")
    systolic_run_cycles = int(counters.get("systolic_run_cycles", 0))
    systolic_op_cycles = int(counters.get("systolic_op_cycles", 0))
    systolic_pe_step_cycles = int(counters.get("systolic_pe_step_cycles", 0))

    pingpong_wait = int(counters.get("pingpong_wait_cycles", 0))
    pingpong_load = int(counters.get("pingpong_load_cycles", 0))
    pingpong_overlap = int(counters.get("pingpong_overlap_cycles", 0))

    sec = total_cycles / clk_hz if clk_hz else None
    return {
        "total_cycles": total_cycles,
        "compute_busy_cycles": compute_cycles,
        "clock_frequency_hz": clk_hz,
        "clock_frequency_mhz": (clk_hz / 1_000_000.0) if clk_hz else None,
        "total_time_s": sec,
        "mac_ops": mac_ops,
        "end_to_end_mac_per_cycle": (mac_ops / total_cycles) if total_cycles else 0.0,
        "busy_mac_per_cycle": (mac_ops / compute_cycles) if compute_cycles else 0.0,
        "systolic_run_cycles": systolic_run_cycles,
        "systolic_op_cycles": systolic_op_cycles,
        "systolic_pe_step_cycles": systolic_pe_step_cycles,
        "systolic_run_mac_per_cycle": (mac_ops / systolic_run_cycles) if systolic_run_cycles else 0.0,
        "systolic_op_mac_per_cycle": (mac_ops / systolic_op_cycles) if systolic_op_cycles else 0.0,
        "systolic_mac_per_pe_step": (mac_ops / systolic_pe_step_cycles) if systolic_pe_step_cycles else 0.0,
        "rtl_peak_mac_per_cycle": RTL_SYSTOLIC_PEAK_MAC_PER_CYCLE,
        "theory_peak_mac_per_cycle": THEORY_PROFILING_PEAK_MAC_PER_CYCLE,
        "systolic_op_util_vs_rtl_peak": ((mac_ops / systolic_op_cycles) / RTL_SYSTOLIC_PEAK_MAC_PER_CYCLE) if systolic_op_cycles else 0.0,
        "systolic_op_util_vs_theory_peak": ((mac_ops / systolic_op_cycles) / THEORY_PROFILING_PEAK_MAC_PER_CYCLE) if systolic_op_cycles else 0.0,
        "ddr_read_words": ddr_read_words,
        "raw_ddr_read_words": raw_ddr_read_words,
        "dma_loader_words": dma_loader_words,
        "ddr_write_words": ddr_write_words,
        "ddr_read_transactions": ddr_read_transactions,
        "ddr_write_transactions": ddr_write_transactions,
        "ddr_transactions": ddr_transactions,
        "ddr_bytes": ddr_bytes,
        "ddr_32B_accesses": math.ceil(ddr_bytes / 32) if ddr_bytes else 0,
        "ddr_read_bytes_per_transaction": ((4 * ddr_read_words) / ddr_read_transactions) if ddr_read_transactions else 0.0,
        "ddr_write_bytes_per_transaction": ((4 * ddr_write_words) / ddr_write_transactions) if ddr_write_transactions else 0.0,
        "ddr_bytes_per_transaction": (ddr_bytes / ddr_transactions) if ddr_transactions else 0.0,
        "ddr_cycles_per_transaction": (ddr_access_cycles / ddr_transactions) if ddr_transactions else 0.0,
        "ddr_cycles_per_32B_access": (ddr_access_cycles / math.ceil(ddr_bytes / 32)) if ddr_bytes else 0.0,
        "DRAM_access_time_cycles": ddr_access_cycles,
        "ddr_load_cycles": ddr_load_cycles,
        "ddr_store_cycles": ddr_store_cycles,
        "bram_read_words": bram_read_words,
        "bram_write_words": bram_write_words,
        "bram_bytes": bram_bytes,
        "BRAM_access_time_cycles": bram_access_cycles,
        "BRAM_bandwidth_bytes_per_cycle": (bram_bytes / bram_access_cycles) if bram_access_cycles else 0.0,
        "tile_stall_cycles": int(counters.get("tile_stall_cycles", 0)),
        "overlap_cycles": int(counters.get("overlap_cycles", 0)),
        "pingpong_wait_cycles": pingpong_wait,
        "pingpong_load_cycles": pingpong_load,
        "pingpong_overlap_cycles": pingpong_overlap,
        "pingpong_overlap_ratio": (pingpong_overlap / pingpong_load) if pingpong_load else 0.0,
    }


## Vectorless Power Report Parsing

Vivado vectorless report 的數值是估計，不是 SAIF/VCD activity 的實測。`DRAM_ENERGY_PER_32B_ACCESS_J` 通常需要由 DDR datasheet、板子 power model 或量測補上。


In [10]:
VIVADO_CORE_DYNAMIC_POWER_W = None
VIVADO_BRAM_DYNAMIC_POWER_W = None
VIVADO_LEAKAGE_POWER_W = None
DRAM_ENERGY_PER_32B_ACCESS_J = None


def _first_float(pattern, text, flags=re.IGNORECASE):
    m = re.search(pattern, text, flags)
    return float(m.group(1)) if m else None


def parse_vivado_power_report(path=REPORT_DIR / "power_impl_vectorless.rpt"):
    path = Path(path)
    if not path.exists():
        print("Power report not found:", path)
        return {}
    text = path.read_text(errors="ignore")

    result = {}
    result["total_on_chip_power_w"] = _first_float(r"Total On-Chip Power\s*\(W\)\s*[:|]?\s*([0-9.]+)", text)
    result["dynamic_power_w"] = _first_float(r"Dynamic\s*\(W\)\s*[:|]?\s*([0-9.]+)", text)
    result["device_static_w"] = _first_float(r"Device Static\s*\(W\)\s*[:|]?\s*([0-9.]+)", text)

    bram_matches = re.findall(r"(?:Block RAM|BRAM|RAMB36|RAMB18)[^\n|]*[|\s]+([0-9]+\.[0-9]+)\s*(?:\||$)", text, re.IGNORECASE)
    result["bram_dynamic_power_w"] = float(bram_matches[-1]) if bram_matches else None
    return result


def load_power_constants(report_path=REPORT_DIR / "power_impl_vectorless.rpt"):
    global VIVADO_CORE_DYNAMIC_POWER_W, VIVADO_BRAM_DYNAMIC_POWER_W, VIVADO_LEAKAGE_POWER_W
    p = parse_vivado_power_report(report_path)
    dyn = p.get("dynamic_power_w")
    bram = p.get("bram_dynamic_power_w")
    static = p.get("device_static_w")
    if dyn is not None:
        VIVADO_CORE_DYNAMIC_POWER_W = dyn
    if bram is not None:
        VIVADO_BRAM_DYNAMIC_POWER_W = bram
    if static is not None:
        VIVADO_LEAKAGE_POWER_W = static
    return p


def _safe_div(num, den):
    return None if den in (0, None) or num is None else num / den


def estimate_energy(counters, clk_hz=PL_CLK_HZ):
    s = summarize_perf(counters, clk_hz)
    time_s = s["total_time_s"]
    core_energy_j = None if VIVADO_CORE_DYNAMIC_POWER_W is None or time_s is None else VIVADO_CORE_DYNAMIC_POWER_W * time_s
    bram_energy_j = None if VIVADO_BRAM_DYNAMIC_POWER_W is None or time_s is None else VIVADO_BRAM_DYNAMIC_POWER_W * time_s
    leakage_energy_j = None if VIVADO_LEAKAGE_POWER_W is None or time_s is None else VIVADO_LEAKAGE_POWER_W * time_s
    dram_accesses_32b = s["ddr_32B_accesses"]
    dram_energy_j = None if DRAM_ENERGY_PER_32B_ACCESS_J is None else dram_accesses_32b * DRAM_ENERGY_PER_32B_ACCESS_J

    return {
        **s,
        "core_dynamic_energy_j": core_energy_j,
        "bram_dynamic_energy_j": bram_energy_j,
        "dram_energy_j": dram_energy_j,
        "leakage_energy_j": leakage_energy_j,
        "Energy_per_macs_J": _safe_div(core_energy_j, s["mac_ops"]),
        "Energy_per_BRAM_access_J": _safe_div(bram_energy_j, s["bram_read_words"] + s["bram_write_words"]),
        "Energy_per_DRAM_access_J": None if DRAM_ENERGY_PER_32B_ACCESS_J is None else DRAM_ENERGY_PER_32B_ACCESS_J,
        "Leakage_Power_W": VIVADO_LEAKAGE_POWER_W,
        "power_constants": {
            "VIVADO_CORE_DYNAMIC_POWER_W": VIVADO_CORE_DYNAMIC_POWER_W,
            "VIVADO_BRAM_DYNAMIC_POWER_W": VIVADO_BRAM_DYNAMIC_POWER_W,
            "VIVADO_LEAKAGE_POWER_W": VIVADO_LEAKAGE_POWER_W,
            "DRAM_ENERGY_PER_32B_ACCESS_J": DRAM_ENERGY_PER_32B_ACCESS_J,
        },
    }


def print_perf_metrics(counters, clk_hz=PL_CLK_HZ):
    m = estimate_energy(counters, clk_hz)
    fields = [
        "total_cycles",
        "compute_busy_cycles",
        "clock_frequency_hz",
        "clock_frequency_mhz",
        "mac_ops",
        "end_to_end_mac_per_cycle",
        "busy_mac_per_cycle",
        "systolic_run_cycles",
        "systolic_op_cycles",
        "systolic_pe_step_cycles",
        "systolic_run_mac_per_cycle",
        "systolic_op_mac_per_cycle",
        "systolic_mac_per_pe_step",
        "rtl_peak_mac_per_cycle",
        "theory_peak_mac_per_cycle",
        "systolic_op_util_vs_rtl_peak",
        "systolic_op_util_vs_theory_peak",
        "BRAM_bandwidth_bytes_per_cycle",
        "BRAM_access_time_cycles",
        "DRAM_access_time_cycles",
        "ddr_read_transactions",
        "ddr_write_transactions",
        "ddr_transactions",
        "ddr_read_bytes_per_transaction",
        "ddr_write_bytes_per_transaction",
        "ddr_bytes_per_transaction",
        "ddr_cycles_per_transaction",
        "ddr_cycles_per_32B_access",
        "ddr_32B_accesses",
        "Energy_per_macs_J",
        "Energy_per_BRAM_access_J",
        "Energy_per_DRAM_access_J",
        "Leakage_Power_W",
        "pingpong_wait_cycles",
        "pingpong_load_cycles",
        "pingpong_overlap_cycles",
        "pingpong_overlap_ratio",
    ]
    for k in fields:
        print(f"{k}: {m.get(k)}")
    return m


In [11]:
FULL_VIT_BLOCKS = 12

CHECKPOINT_CANDIDATES = [
    HOST_GOLDEN_ROOT / "rms_qat_best.pt",
    GOLDEN_ROOT.parent.parent / "rms_qat_best.pt",
    NOTEBOOK_DIR / "rms_qat_best.pt",
]
IMAGE_CANDIDATES = [
    HOST_GOLDEN_ROOT / "n01440764_10194.jpg",
    GOLDEN_ROOT.parent.parent / "n01440764_10194.jpg",
    NOTEBOOK_DIR / "n01440764_10194.jpg",
]
GOLDEN_GEN_CANDIDATES = [
    HOST_GOLDEN_ROOT / "golden_gen.py",
    GOLDEN_ROOT.parent.parent / "golden_gen.py",
    NOTEBOOK_DIR / "golden_gen.py",
]
FULL_VIT_CACHE = GOLDEN_ROOT / "full_vit_block12_integer_golden.npz"
HARDWARE_ROOT_CANDIDATES = [
    Path("C:/\u6210\u5927\u8c93/\u5927\u56db/AOC/\u70b8\u5f48\u60e1\u9b54/AOC_final_project-main/hardware"),
    NOTEBOOK_DIR / "hardware",
    NOTEBOOK_DIR.parent / "hardware",
    Path.cwd(),
]


def first_existing_path(paths, label):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    tried = "\n".join(str(Path(p)) for p in paths)
    raise FileNotFoundError(f"Cannot find {label}. Tried:\n{tried}")


def import_golden_gen_module():
    import importlib.util
    import os
    if not os.environ.get("VIT_HARDWARE_ROOT"):
        for hw in HARDWARE_ROOT_CANDIDATES:
            hw = Path(hw)
            if (hw / "Streaming_RMSNorm_Unit").exists() and (hw / "PPU" / "GELU_Unit.sv").exists():
                os.environ["VIT_HARDWARE_ROOT"] = str(hw)
                break
    path = first_existing_path(GOLDEN_GEN_CANDIDATES, "golden_gen.py")
    spec = importlib.util.spec_from_file_location("vit_real_model_golden_gen", path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def imagenet_labels():
    candidates = [
        GOLDEN_ROOT / "imagenet_classes.txt",
        HOST_GOLDEN_ROOT / "imagenet_classes.txt",
        NOTEBOOK_DIR / "imagenet_classes.txt",
        GOLDEN_ROOT / "imagenet_class_index.json",
        HOST_GOLDEN_ROOT / "imagenet_class_index.json",
        NOTEBOOK_DIR / "imagenet_class_index.json",
    ]
    for p in candidates:
        if not p.exists():
            continue
        if p.suffix.lower() == ".txt":
            lines = [x.strip() for x in p.read_text(encoding="utf-8").splitlines() if x.strip()]
            if len(lines) >= 1000:
                return lines[:1000]
        if p.suffix.lower() == ".json":
            obj = json.loads(p.read_text(encoding="utf-8"))
            labels = [None] * 1000
            for k, v in obj.items():
                idx = int(k)
                if isinstance(v, (list, tuple)) and len(v) >= 2:
                    labels[idx] = str(v[1])
                else:
                    labels[idx] = str(v)
            if all(x is not None for x in labels):
                return labels
    return [f"class_{i}" for i in range(1000)]


def top5_from_logits(logits, labels=None):
    logits = np.asarray(logits, dtype=np.float64).reshape(-1)
    labels = labels or imagenet_labels()
    idx = np.argsort(logits)[::-1][:5]
    return [
        {
            "rank": r + 1,
            "class_id": int(i),
            "label": labels[int(i)] if int(i) < len(labels) else f"class_{int(i)}",
            "score": float(logits[int(i)]),
        }
        for r, i in enumerate(idx)
    ]


def print_top5(title, rows):
    print(title)
    for row in rows:
        print(f"  {row['rank']}. id={row['class_id']:4d}  score={row['score']: .6f}  {row['label']}")


def rmsnorm_hw_numpy(x_q, gamma_q, inv_lut):
    x_s = np.asarray(x_q, dtype=np.int64) - 128
    sum_sq = np.sum(x_s * x_s, axis=1)
    idx = ((sum_sq.astype(np.int64) * 620 + (1 << 15)) >> 16)
    idx = np.clip(idx, 0, 1023)
    inv = np.asarray(inv_lut, dtype=np.int64)[idx]
    prod = x_s * inv[:, None] * np.asarray(gamma_q, dtype=np.int64)[None, :]
    y_s = np.right_shift(prod, 28)
    return np.clip(np.clip(y_s, -128, 127) + 128, 0, 255).astype(np.uint8)


def _build_patch_input_q(gg, state, image_u8, patch_shift=None):
    patches_u8 = gg.patches_from_image(image_u8)
    cls = gg.t(state, "cls_token").reshape(1, EMBED_DIM)[0]
    pos = gg.t(state, "pos_embed").reshape(TOKEN_NUM, EMBED_DIM)
    patch_w = gg.t(state, "patch_embed.proj.weight")
    patch_b = gg.t(state, "patch_embed.proj.bias")

    patch_w_eff = np.zeros((gg.PATCH_ELEMS, EMBED_DIM), dtype=np.float64)
    patch_b_eff = patch_b.astype(np.float64).copy()
    for y in range(PATCH_SIZE):
        for x in range(PATCH_SIZE):
            for c in range(IMG_C):
                k = (y * PATCH_SIZE + x) * IMG_C + c
                w_oc = patch_w[:, c, y, x]
                patch_w_eff[k, :] = w_oc / (255.0 * gg.IMAGENET_STD[c])
                patch_b_eff -= w_oc * (gg.IMAGENET_MEAN[c] / gg.IMAGENET_STD[c])

    patch_float = patches_u8.astype(np.float64) @ patch_w_eff + patch_b_eff[None, :]
    x_float = np.zeros((TOKEN_NUM, EMBED_DIM), dtype=np.float64)
    x_float[0] = cls + pos[0]
    x_float[1:] = patch_float + pos[1:]
    x_scale = float(max(np.max(np.abs(x_float)) / 120.0, 1e-6))

    if patch_shift is None:
        patch_shift = int(load_case_manifest().get("patch_requant_shift", 0))
    patch_min_w_scale = float(np.max(np.abs(patch_w_eff)) / 127.0) if np.max(np.abs(patch_w_eff)) else 1e-9
    patch_w_scale = max(x_scale / (1 << int(patch_shift)), patch_min_w_scale)
    if patch_w_scale != x_scale / (1 << int(patch_shift)):
        x_scale = patch_w_scale * (1 << int(patch_shift))

    patch_w_q, _ = gg.quant_s8(patch_w_eff, patch_w_scale)
    patch_b_q = np.rint(patch_b_eff / patch_w_scale).astype(np.int64)
    patch_psum = patches_u8.astype(np.int64) @ patch_w_q.astype(np.int64) + patch_b_q[None, :]
    patch_q = gg.requant_zp128(patch_psum, int(patch_shift))
    pos_q = gg.quant_u8_zp128(pos, x_scale)
    cls_q = gg.quant_u8_zp128(cls, x_scale)

    x_q = np.zeros((TOKEN_NUM, EMBED_DIM), dtype=np.uint8)
    x_q[0] = gg.residual_add(cls_q[None, :], pos_q[0:1])[0]
    x_q[1:] = gg.residual_add(patch_q, pos_q[1:])
    return x_q, x_scale, int(patch_shift)


def _run_one_integer_block(gg, state, x_q, x_scale, block_idx, inv_lut, exp_lut, gelu_rom, fc1_weight_scale=1.0 / 1024.0):
    prefix = f"blocks.{int(block_idx)}"
    gamma1_q = np.clip(np.rint(gg.t(state, f"{prefix}.norm1.weight") * (1 << 14)), -32768, 32767).astype(np.int64)
    gamma2_q = np.clip(np.rint(gg.t(state, f"{prefix}.norm2.weight") * (1 << 14)), -32768, 32767).astype(np.int64)

    x_norm = gg.rmsnorm_hw(x_q, gamma1_q, inv_lut)

    qkv_scale = 2.0 ** -4
    qkv_w_scale = qkv_scale / (1 << gg.SHIFT_QKV)
    qkv_w_q, _ = gg.quant_s8(gg.t(state, f"{prefix}.attn.qkv.weight"), qkv_w_scale)
    qkv_b_q = np.rint(gg.t(state, f"{prefix}.attn.qkv.bias") / qkv_w_scale).astype(np.int64)
    q_q, k_q, v_q, _ = gg.qkv_hw(x_norm, qkv_w_q, qkv_b_q)

    q_s = q_q.astype(np.int64) - 128
    k_s = k_q.astype(np.int64) - 128
    v_s = v_q.astype(np.int64) - 128
    scores = np.zeros((gg.HEAD_NUM, TOKEN_NUM, TOKEN_NUM), dtype=np.int64)
    for h in range(gg.HEAD_NUM):
        sl = slice(h * gg.HEAD_DIM, (h + 1) * gg.HEAD_DIM)
        scores[h] = q_s[:, sl] @ k_s[:, sl].T
    attn_q = gg.softmax_hw(scores, exp_lut)[:, :, :TOKEN_NUM]

    o_heads = np.zeros((TOKEN_NUM, EMBED_DIM), dtype=np.uint8)
    for h in range(gg.HEAD_NUM):
        sl = slice(h * gg.HEAD_DIM, (h + 1) * gg.HEAD_DIM)
        psum = attn_q[h].astype(np.int64) @ v_s[:, sl].astype(np.int64)
        o_heads[:, sl] = gg.requant_zp128(psum, gg.SHIFT_ATTN_V)
    o_scale = (1.0 / 128.0) * qkv_scale * (1 << gg.SHIFT_ATTN_V)

    out_w_scale = x_scale / (o_scale * (1 << gg.SHIFT_OUT_PROJ))
    out_w_q, _ = gg.quant_s8(gg.t(state, f"{prefix}.attn.proj.weight"), out_w_scale)
    out_b_q = np.rint(gg.t(state, f"{prefix}.attn.proj.bias") / (o_scale * out_w_scale)).astype(np.int64)
    out_q, _ = gg.linear_hw(o_heads, out_w_q, out_b_q, gg.SHIFT_OUT_PROJ, act_zp=128)
    x_mid = gg.residual_add(out_q, x_q)

    x_mid_norm = gg.rmsnorm_hw(x_mid, gamma2_q, inv_lut)

    fc1_w_q, _ = gg.quant_s8(gg.t(state, f"{prefix}.mlp.fc1.weight"), float(fc1_weight_scale))
    fc1_b_q = np.rint(gg.t(state, f"{prefix}.mlp.fc1.bias") / float(fc1_weight_scale)).astype(np.int64)
    _, fc1_psum = gg.linear_hw(x_mid_norm, fc1_w_q, fc1_b_q, gg.SHIFT_FC1, act_zp=128)
    gelu_q = gg.gelu_requant_hw(fc1_psum, gelu_rom)

    fc2_w_scale = x_scale / (1.0 * (1 << gg.SHIFT_FC2))
    fc2_w_q, _ = gg.quant_s8(gg.t(state, f"{prefix}.mlp.fc2.weight"), fc2_w_scale)
    fc2_b_q = np.rint(gg.t(state, f"{prefix}.mlp.fc2.bias") / fc2_w_scale).astype(np.int64)
    fc2_q, _ = gg.linear_hw(gelu_q, fc2_w_q, fc2_b_q, gg.SHIFT_FC2, act_zp=128)
    x_out = gg.residual_add(fc2_q, x_mid)
    return x_out


def _classifier_logits_from_final_x_u8(x_final_u8, gg=None, state=None, inv_lut=None, cache=None):
    x_final_u8 = np.asarray(x_final_u8, dtype=np.uint8).reshape(TOKEN_NUM, EMBED_DIM)
    if cache is not None and "head_w" in cache:
        final_norm_gamma_q = cache["final_norm_gamma_q"].astype(np.int64)
        head_w = cache["head_w"].astype(np.float64)
        head_b = cache["head_b"].astype(np.float64)
        if inv_lut is None:
            inv_lut = cache["inv_lut"].astype(np.int64)
        cls_norm_q = rmsnorm_hw_numpy(x_final_u8[0:1], final_norm_gamma_q, inv_lut)[0]
    else:
        if gg is None:
            gg = import_golden_gen_module()
        if state is None:
            state = gg.load_state(first_existing_path(CHECKPOINT_CANDIDATES, "checkpoint"))
        if inv_lut is None:
            inv_lut = gg.load_hex_u16(gg.INV_LUT_HEX)
        final_norm_gamma_q = np.clip(np.rint(gg.t(state, "norm.weight") * (1 << 14)), -32768, 32767).astype(np.int64)
        head_w = gg.t(state, "head.weight").astype(np.float64)
        head_b = gg.t(state, "head.bias").astype(np.float64) if "head.bias" in state else np.zeros(head_w.shape[0], dtype=np.float64)
        cls_norm_q = gg.rmsnorm_hw(x_final_u8[0:1], final_norm_gamma_q, inv_lut)[0]

    cls_feature = cls_norm_q.astype(np.float64) - 128.0
    logits = cls_feature @ head_w.T + head_b
    return logits, cls_norm_q


def compute_full_vit_integer_golden(force=False, save_cache=True):
    if FULL_VIT_CACHE.exists() and not force:
        data = dict(np.load(FULL_VIT_CACHE, allow_pickle=False))
        return data

    gg = import_golden_gen_module()
    checkpoint = first_existing_path(CHECKPOINT_CANDIDATES, "checkpoint")
    image_path = first_existing_path(IMAGE_CANDIDATES, "real input image")
    state = gg.load_state(checkpoint)
    image_u8 = gg.load_image_u8(image_path)
    inv_lut = gg.load_hex_u16(gg.INV_LUT_HEX)
    exp_lut = gg.load_hex_u16(gg.EXP_LUT_HEX)
    gelu_rom = gg.load_gelu_rom(gg.GELU_SV)

    x_q, x_scale, patch_shift = _build_patch_input_q(gg, state, image_u8)
    cls_after_each_block = np.zeros((FULL_VIT_BLOCKS, EMBED_DIM), dtype=np.uint8)
    for block_idx in range(FULL_VIT_BLOCKS):
        x_q = _run_one_integer_block(gg, state, x_q, x_scale, block_idx, inv_lut, exp_lut, gelu_rom)
        cls_after_each_block[block_idx] = x_q[0]

    logits, cls_norm_q = _classifier_logits_from_final_x_u8(x_q, gg=gg, state=state, inv_lut=inv_lut)
    head_w = gg.t(state, "head.weight").astype(np.float64)
    head_b = gg.t(state, "head.bias").astype(np.float64) if "head.bias" in state else np.zeros(head_w.shape[0], dtype=np.float64)
    final_norm_gamma_q = np.clip(np.rint(gg.t(state, "norm.weight") * (1 << 14)), -32768, 32767).astype(np.int64)

    out = {
        "x_final_u8": x_q.astype(np.uint8),
        "cls_u8": x_q[0].astype(np.uint8),
        "cls_norm_u8": cls_norm_q.astype(np.uint8),
        "cls_after_each_block_u8": cls_after_each_block,
        "logits": logits.astype(np.float64),
        "x_scale": np.asarray([x_scale], dtype=np.float64),
        "patch_shift": np.asarray([patch_shift], dtype=np.int64),
        "head_w": head_w.astype(np.float32),
        "head_b": head_b.astype(np.float32),
        "final_norm_gamma_q": final_norm_gamma_q.astype(np.int16),
        "inv_lut": inv_lut.astype(np.int64),
    }
    if save_cache:
        GOLDEN_ROOT.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(FULL_VIT_CACHE, **out)
        top5_rows = top5_from_logits(out["logits"])
        (GOLDEN_ROOT / "full_vit_block12_top5.json").write_text(
            json.dumps(top5_rows, indent=2, ensure_ascii=False) + "\n",
            encoding="utf-8",
        )
        print("Saved full-ViT golden cache:", FULL_VIT_CACHE)
    return out


def xout_words_to_token_matrix(out_words):
    words = np.asarray(out_words, dtype=np.uint32).reshape(-1)
    if len(words) < XOUT_ELEMS:
        raise ValueError(f"Need at least {XOUT_ELEMS} X_out words, got {len(words)}")
    return (words[:XOUT_ELEMS] & np.uint32(0xFF)).astype(np.uint8).reshape(TOKEN_NUM, EMBED_DIM)


def report_full_vit_top5(hardware_xout_words=None, hardware_x_final_u8=None, force_recompute_golden=False):
    golden = compute_full_vit_integer_golden(force=force_recompute_golden, save_cache=True)
    labels = imagenet_labels()
    golden_top5 = top5_from_logits(golden["logits"], labels)
    print_top5("Golden 12-block CLS top-5", golden_top5)

    hardware_top5 = None
    if hardware_x_final_u8 is None and hardware_xout_words is not None:
        hardware_x_final_u8 = xout_words_to_token_matrix(hardware_xout_words)

    if hardware_x_final_u8 is not None:
        cache = golden
        logits_hw, cls_norm_hw = _classifier_logits_from_final_x_u8(hardware_x_final_u8, cache=cache)
        hardware_top5 = top5_from_logits(logits_hw, labels)
        print_top5("Hardware 12-block CLS top-5", hardware_top5)
        same_ids = [g["class_id"] == h["class_id"] for g, h in zip(golden_top5, hardware_top5)]
        print("Top-5 same-rank matches:", sum(same_ids), "/ 5")
    else:
        print("Hardware 12-block CLS top-5: no block12 X_out buffer was provided.")
        print("Run run_full_vit_12_blocks_hardware(mmio) after rebuilding RTL with TARGET_X_BUF and RUN_MODE support.")

    return {"golden": golden_top5, "hardware": hardware_top5}



def set_transformer_only(mmio, enable):
    wr(mmio, REG_RUN_MODE, 1 if enable else 0)


def xout_words_to_packed_x_words(out_words):
    x_u8 = xout_words_to_token_matrix(out_words).reshape(-1)
    return pack_u8_to_u32(x_u8)


def block_case_dir(block_idx):
    block_idx = int(block_idx)
    candidates = [
        GOLDEN_ROOT / "blocks" / f"block_{block_idx:02d}",
        HOST_GOLDEN_ROOT / "hex" / GOLDEN_CASE_NAME / "blocks" / f"block_{block_idx:02d}",
        NOTEBOOK_DIR / "golden_gen" / "hex" / GOLDEN_CASE_NAME / "blocks" / f"block_{block_idx:02d}",
    ]
    for p in candidates:
        if (p / "gamma.hex").exists() and (p / "transformer_weight.hex").exists() and (p / "transformer_bias.hex").exists():
            return p
    tried = "\n".join(str(p) for p in candidates)
    raise FileNotFoundError(
        f"Missing block {block_idx:02d} vectors. Run golden_gen.py again; it now creates blocks/block_00..block_11. Tried:\n{tried}"
    )


def load_block_loader_words(block_idx):
    d = block_case_dir(block_idx)
    gamma_words = gamma_words_for_loader(read_hex_words(d / "gamma.hex"))
    trans_weight_words = read_hex_words(d / "transformer_weight.hex")
    trans_bias_words = read_hex_words(d / "transformer_bias.hex")
    manifest_p = d / "manifest.json"
    manifest = json.loads(manifest_p.read_text(encoding="utf-8")) if manifest_p.exists() else {"block_index": int(block_idx)}
    return {
        "dir": d,
        "manifest": manifest,
        "gamma": gamma_words,
        "trans_weight": trans_weight_words,
        "trans_bias": trans_bias_words,
    }


def _sum_counter_dicts(dicts):
    out = {name: 0 for name in PERF_REGS.keys()}
    for d in dicts:
        for k, v in d.items():
            out[k] = int(out.get(k, 0)) + int(v)
    return out


def _run_configured_transformer_block(
    mmio,
    block_idx,
    block_words,
    page_buffers,
    patch_weight_src=None,
    patch_bias_src=None,
    transformer_only=False,
    patch_shift=None,
    verbose_pages=False,
    timeout_s=180.0,
):
    clear_done(mmio)
    if patch_shift is not None:
        wr(mmio, REG_PATCH_SHIFT, int(patch_shift))
    set_transformer_only(mmio, transformer_only)

    ddr_load(mmio, TARGET_GAMMA, block_words["gamma"], local_base=0, timeout_s=timeout_s)
    trans_weight_buf = alloc_u32(block_words["trans_weight"])
    trans_bias_buf = alloc_u32(block_words["trans_bias"])

    load_requested_tiles_once(
        mmio,
        patch_weight_src,
        patch_bias_src,
        trans_weight_buf,
        trans_bias_buf,
    )
    start_compute(mmio)
    service_tiles_until_done(
        mmio,
        patch_weight_src,
        patch_bias_src,
        trans_weight_buf,
        trans_bias_buf,
        page_buffers=page_buffers,
        timeout_s=timeout_s,
        verbose_pages=verbose_pages,
    )
    out_buf = ddr_store_xout(mmio, XOUT_ELEMS, local_base=0, timeout_s=timeout_s)
    counters = read_perf_counters(mmio)
    return out_buf, counters


def run_full_vit_12_blocks_hardware(mmio, patch_shift=None, verbose_pages=False, tolerance=0, timeout_s=180.0):
    """Run the same one-block RTL core 12 times from Python.

    Block 0 uses the normal raw-image -> patch embedding -> transformer path.
    Blocks 1..11 load previous X_out back into X BRAM through TARGET_X_BUF and
    start the RTL in transformer-only mode through REG_RUN_MODE bit0.
    """
    case_manifest = load_case_manifest()
    if patch_shift is None:
        patch_shift = int(case_manifest.get("patch_requant_shift", 0))
    wr(mmio, REG_PATCH_SHIFT, patch_shift)

    print("Preparing static image/pos/cls buffers")
    static_buffers = load_static_inputs(mmio)
    page_buffers = {
        "image": static_buffers["image"],
        "pos": static_buffers["pos"],
        "gelu": alloc_gelu_buffer(),
    }

    root_words = load_full_design_words()
    patch_buffers = alloc_weight_buffers({
        "patch_weight": root_words.get("patch_weight"),
        "patch_bias": root_words.get("patch_bias"),
    })
    patch_weight_src = patch_buffers.get("patch_weight")
    patch_bias_src = patch_buffers.get("patch_bias")
    if patch_weight_src is None or patch_bias_src is None:
        raise FileNotFoundError("Patch embedding weight/bias are required for block0.")

    out_buf = None
    block_results = []
    block_counters = []

    for block_idx in range(FULL_VIT_BLOCKS):
        print(f"\n=== Hardware block {block_idx:02d}/{FULL_VIT_BLOCKS - 1:02d} ===")
        clear_done(mmio)
        clear_perf(mmio)
        block_words = load_block_loader_words(block_idx)
        page_buffers["gelu"] = alloc_gelu_buffer()

        if block_idx == 0:
            set_transformer_only(mmio, False)
            out_buf, counters = _run_configured_transformer_block(
                mmio,
                block_idx,
                block_words,
                page_buffers,
                patch_weight_src=patch_weight_src,
                patch_bias_src=patch_bias_src,
                transformer_only=False,
                patch_shift=patch_shift,
                verbose_pages=verbose_pages,
                timeout_s=timeout_s,
            )
        else:
            x_words = xout_words_to_packed_x_words(out_buf)
            x_buf = ddr_load(mmio, TARGET_X_BUF, x_words, local_base=0, timeout_s=timeout_s)
            set_transformer_only(mmio, True)
            out_buf, counters = _run_configured_transformer_block(
                mmio,
                block_idx,
                block_words,
                page_buffers,
                patch_weight_src=None,
                patch_bias_src=None,
                transformer_only=True,
                patch_shift=patch_shift,
                verbose_pages=verbose_pages,
                timeout_s=timeout_s,
            )
            block_words["x_input_buffer"] = x_buf

        block_counters.append(counters)
        block_result = {
            "block": block_idx,
            "vector_dir": str(block_words["dir"]),
            "counters": counters,
            "perf": summarize_perf(counters, clk_hz=PL_CLK_HZ),
        }
        block_results.append(block_result)
        print("Block perf:", block_result["perf"])

    set_transformer_only(mmio, False)
    total_counters = _sum_counter_dicts(block_counters)
    hardware_x_final_u8 = xout_words_to_token_matrix(out_buf)
    golden = compute_full_vit_integer_golden(force=False, save_cache=True)
    verify_result, _, _, _ = compare_xout_arrays(
        hardware_x_final_u8.reshape(-1),
        golden["x_final_u8"].reshape(-1),
        tolerance=tolerance,
    )
    print("\n12-block X_final verify:")
    print(f"  exact match      : {verify_result['exact_match_percent']:.4f}%")
    print(f"  mismatch         : {verify_result['mismatch_percent']:.4f}%  ({verify_result['mismatch_count']} / {verify_result['total_values']})")
    print(f"  mean abs diff    : {verify_result['mean_abs_diff_lsb']:.4f} LSB ({verify_result['mean_abs_diff_percent_of_255']:.4f}% of 255)")
    print(f"  max abs diff     : {verify_result['max_abs_diff']} LSB")
    print("\n12-block accumulated counters:", total_counters)
    print("12-block accumulated perf:", summarize_perf(total_counters, clk_hz=PL_CLK_HZ))

    run_full_vit_12_blocks_hardware.last_blocks = block_results
    run_full_vit_12_blocks_hardware.last_counters = total_counters
    run_full_vit_12_blocks_hardware.last_verify = verify_result
    run_full_vit_12_blocks_hardware.last_x_final_u8 = hardware_x_final_u8
    return out_buf, total_counters, block_results


## Stage Checkpoint Per-layer Verification

This cell group uses RTL stage checkpoint registers to pause after each major stage, dump the matching buffer, compare it with software golden, and print per-stage counters.


In [12]:

STAGE_CHECKPOINT_DRIVER_VERSION = "2026-06-25-stage-checkpoint-v18-rtl-physical-golden"

REG_STAGE_CONTROL = globals().get("REG_STAGE_CONTROL", 0x0A0)
REG_STAGE_STATUS  = globals().get("REG_STAGE_STATUS",  0x0A4)
REG_STAGE_TOTAL   = globals().get("REG_STAGE_TOTAL",   0x0A8)
REG_STAGE_BUSY    = globals().get("REG_STAGE_BUSY",    0x0AC)
REG_STAGE_DDR_LD  = globals().get("REG_STAGE_DDR_LD",  0x0B0)
REG_STAGE_DDR_ST  = globals().get("REG_STAGE_DDR_ST",  0x0B4)
REG_STAGE_DDR_RDW = globals().get("REG_STAGE_DDR_RDW", 0x0B8)
REG_STAGE_DDR_WRW = globals().get("REG_STAGE_DDR_WRW", 0x0BC)
REG_STAGE_BRAM_RDW = globals().get("REG_STAGE_BRAM_RDW", 0x0C0)
REG_STAGE_BRAM_WRW = globals().get("REG_STAGE_BRAM_WRW", 0x0C4)
REG_STAGE_BRAM_CYC = globals().get("REG_STAGE_BRAM_CYC", 0x0C8)
REG_STAGE_MAC_LO   = globals().get("REG_STAGE_MAC_LO",   0x0CC)
REG_STAGE_MAC_HI   = globals().get("REG_STAGE_MAC_HI",   0x0D0)
REG_STAGE_PP_WAIT  = globals().get("REG_STAGE_PP_WAIT",  0x0D4)
REG_STAGE_PP_LOAD  = globals().get("REG_STAGE_PP_LOAD",  0x0D8)
REG_STAGE_PP_OVLP  = globals().get("REG_STAGE_PP_OVLP",  0x0DC)
REG_STAGE_SYS_RUN  = globals().get("REG_STAGE_SYS_RUN",  0x0EC)
REG_STAGE_SYS_OP   = globals().get("REG_STAGE_SYS_OP",   0x0F0)
REG_STAGE_SYS_STEP = globals().get("REG_STAGE_SYS_STEP", 0x0F4)
REG_STAGE_DDR_RTXN = globals().get("REG_STAGE_DDR_RTXN", 0x118)
REG_STAGE_DDR_WTXN = globals().get("REG_STAGE_DDR_WTXN", 0x11C)
TARGET_NORM_BUF   = globals().get("TARGET_NORM_BUF", 11)
TARGET_V_XMID_BUF = globals().get("TARGET_V_XMID_BUF", 12)
TARGET_SHARED_BUF = globals().get("TARGET_SHARED_BUF", 13)

STAGE_CTRL_ENABLE = 1 << 0
STAGE_CTRL_RESUME = 1 << 1
STAGE_CTRL_CLEAR  = 1 << 2
STAGE_STATUS_PENDING = 1 << 31
STAGE_STATUS_DONE    = 1 << 30

STAGE_COUNTER_REGS = {
    "total_cycles": REG_STAGE_TOTAL,
    "compute_busy_cycles": REG_STAGE_BUSY,
    "ddr_load_cycles": REG_STAGE_DDR_LD,
    "ddr_store_cycles": REG_STAGE_DDR_ST,
    "ddr_read_words": REG_STAGE_DDR_RDW,
    "ddr_write_words": REG_STAGE_DDR_WRW,
    "ddr_read_transactions": REG_STAGE_DDR_RTXN,
    "ddr_write_transactions": REG_STAGE_DDR_WTXN,
    "bram_read_words": REG_STAGE_BRAM_RDW,
    "bram_write_words": REG_STAGE_BRAM_WRW,
    "bram_access_cycles": REG_STAGE_BRAM_CYC,
    "mac_ops_lo": REG_STAGE_MAC_LO,
    "mac_ops_hi": REG_STAGE_MAC_HI,
    "pingpong_wait_cycles": REG_STAGE_PP_WAIT,
    "pingpong_load_cycles": REG_STAGE_PP_LOAD,
    "pingpong_overlap_cycles": REG_STAGE_PP_OVLP,
    "systolic_run_cycles": REG_STAGE_SYS_RUN,
    "systolic_op_cycles": REG_STAGE_SYS_OP,
    "systolic_pe_step_cycles": REG_STAGE_SYS_STEP,
}

def activation_physical_word_count(channel_num, token_num=TOKEN_NUM):
    token_tiles = (int(token_num) + TOKEN_TILE - 1) // TOKEN_TILE
    channel_tiles = (int(channel_num) + CHANNEL_TILE - 1) // CHANNEL_TILE
    return token_tiles * channel_tiles * TOKEN_TILE * TILE_ROW_WORDS


ACT_PHYS_WORDS = activation_physical_word_count(EMBED_DIM)
GELU_PHYS_WORDS = activation_physical_word_count(FFN_CHANNEL_NUM)

# Stage dump targets T_X_BUF/T_NORM_BUF/T_VXMID_BUF return RTL physical BRAM words,
# not flat token-major activation bytes.  Therefore Python compares by using the
# same physical address mapping as RTL act_word_index()/gelu_word_index().
STAGE_META = {
    1: {"name": "Patch Embedding", "dump_target": TARGET_X_BUF, "dump_words": ACT_PHYS_WORDS, "golden_key": "x_after_patch", "layout": "activation_phys", "channels": EMBED_DIM},
    2: {"name": "RMSNorm1", "dump_target": TARGET_NORM_BUF, "dump_words": ACT_PHYS_WORDS, "golden_key": "x_norm1", "layout": "activation_phys", "channels": EMBED_DIM},
    3: {"name": "QKV / QK^T / Softmax / A*V", "dump_target": TARGET_NORM_BUF, "dump_words": ACT_PHYS_WORDS, "golden_key": "o_attn", "layout": "activation_phys", "channels": EMBED_DIM},
    5: {"name": "Output Projection + Residual Add 1 / X_mid", "dump_target": TARGET_V_XMID_BUF, "dump_words": ACT_PHYS_WORDS, "golden_key": "x_mid", "layout": "activation_phys", "channels": EMBED_DIM},
    6: {"name": "RMSNorm2", "dump_target": TARGET_NORM_BUF, "dump_words": ACT_PHYS_WORDS, "golden_key": "x_mid_norm", "layout": "activation_phys", "channels": EMBED_DIM},
    7: {"name": "FC1 + GELU", "source": "gelu_ddr_buffer", "dump_words": GELU_PHYS_WORDS, "golden_key": "gelu", "layout": "activation_phys", "channels": FFN_CHANNEL_NUM},
    9: {"name": "FC2 + Residual Add 2 / X_out", "dump_target": TARGET_X_BUF, "dump_words": ACT_PHYS_WORDS, "golden_key": "x_out", "layout": "activation_phys", "channels": EMBED_DIM},
}

def decode_stage_status(word):
    word = int(word)
    return {"pending": bool(word & STAGE_STATUS_PENDING), "done_sticky": bool(word & STAGE_STATUS_DONE), "sequence": (word >> 10) & 0xFF, "phase": (word >> 4) & 0x1F, "stage_id": word & 0xF, "raw": word}

def clear_stage_checkpoint(mmio, enable=True):
    wr(mmio, REG_STAGE_CONTROL, (STAGE_CTRL_ENABLE if enable else 0) | STAGE_CTRL_CLEAR)
    time.sleep(0.001)
    wr(mmio, REG_STAGE_CONTROL, STAGE_CTRL_ENABLE if enable else 0)

def enable_stage_checkpoint(mmio, enable=True):
    wr(mmio, REG_STAGE_CONTROL, STAGE_CTRL_ENABLE if enable else 0)

def resume_stage_checkpoint(mmio, enable=True):
    base = STAGE_CTRL_ENABLE if enable else 0
    wr(mmio, REG_STAGE_CONTROL, base | STAGE_CTRL_RESUME)
    time.sleep(0.001)
    wr(mmio, REG_STAGE_CONTROL, base)

def read_stage_counters(mmio):
    return {name: rd(mmio, off) for name, off in STAGE_COUNTER_REGS.items()}

def stage_counters_for_summary(stage_counters):
    out = {name: 0 for name in PERF_REGS.keys()}
    out.update(stage_counters)
    return out

def unpack_u32_words_to_u8(words, valid_bytes=None):
    words = np.asarray(words, dtype=np.uint32).reshape(-1)
    data = np.empty(words.size * 4, dtype=np.uint8)
    data[0::4] = (words & np.uint32(0xFF)).astype(np.uint8)
    data[1::4] = ((words >> np.uint32(8)) & np.uint32(0xFF)).astype(np.uint8)
    data[2::4] = ((words >> np.uint32(16)) & np.uint32(0xFF)).astype(np.uint8)
    data[3::4] = ((words >> np.uint32(24)) & np.uint32(0xFF)).astype(np.uint8)
    return data[:int(valid_bytes)] if valid_bytes is not None else data

def compare_u8_arrays(rtl_u8, golden_u8, tolerance=0):
    rtl = np.asarray(rtl_u8, dtype=np.uint8).reshape(-1)
    golden = np.asarray(golden_u8, dtype=np.uint8).reshape(-1)
    n = min(len(rtl), len(golden))
    rtl = rtl[:n]
    golden = golden[:n]
    diff = rtl.astype(np.int16) - golden.astype(np.int16)
    abs_diff = np.abs(diff)
    exact = abs_diff == 0
    within = abs_diff <= int(tolerance)
    bad = np.where(~within)[0][:20]
    return {
        "count": int(n),
        "exact_match_percent": float(np.mean(exact) * 100.0) if n else 0.0,
        "within_tolerance_percent": float(np.mean(within) * 100.0) if n else 0.0,
        "max_abs_diff": int(np.max(abs_diff)) if n else 0,
        "mean_abs_diff": float(np.mean(abs_diff)) if n else 0.0,
        "rmse": float(np.sqrt(np.mean(diff.astype(np.float64) ** 2))) if n else 0.0,
        "first_mismatches": [{"index": int(i), "rtl": int(rtl[i]), "golden": int(golden[i]), "diff": int(diff[i])} for i in bad],
    }


def _load_canonical_stage_npy_golden():
    """Load trusted stage outputs generated by the official hex case.

    The canonical files are the .npy files generated together with the hex
    vectors.  They are the same files used by the previous one-block flow.
    A stale stage_golden_block*.npz must not override these files.
    """
    mapping = {
        "x_after_patch": GOLDEN_ROOT / "x_after_patch.npy",
        "x_mid": GOLDEN_ROOT / "x_mid.npy",
        "x_out": GOLDEN_ROOT / "x_out.npy",
    }
    out = {}
    for key, path in mapping.items():
        if path.exists():
            out[key] = np.load(path).astype(np.uint8)
    return out


def _stage_npz_is_consistent(stage_data, canonical_data):
    for key, ref in canonical_data.items():
        if key in stage_data:
            arr = np.asarray(stage_data[key], dtype=np.uint8)
            if arr.shape != ref.shape or not np.array_equal(arr, ref):
                return False
    return True

def compute_one_block_stage_golden(block_idx=0, x_input_u8=None, x_scale=None, patch_shift=None):
    # The official one-block vectors are the .npy/hex files in GOLDEN_ROOT.
    # A stale stage_golden_block*.npz can make per-stage compare look wrong,
    # so only use it when it agrees with the canonical .npy outputs.
    canonical = _load_canonical_stage_npy_golden() if x_input_u8 is None else {}
    stage_npz = GOLDEN_ROOT / f"stage_golden_block{int(block_idx)}.npz"
    if x_input_u8 is None and stage_npz.exists():
        data = np.load(stage_npz, allow_pickle=False)
        stage_data = {k: data[k] for k in data.files}
        if _stage_npz_is_consistent(stage_data, canonical):
            return stage_data
        print(f"WARNING: ignoring stale {stage_npz.name}; it does not match canonical .npy golden files.")
        # Keep only trusted canonical keys.  Missing intermediate stages will be skipped.
        if canonical:
            return canonical

    gg = import_golden_gen_module()
    state = gg.load_state(first_existing_path(CHECKPOINT_CANDIDATES, "checkpoint"))
    inv_lut = gg.load_hex_u16(gg.INV_LUT_HEX)
    exp_lut = gg.load_hex_u16(gg.EXP_LUT_HEX)
    gelu_rom = gg.load_gelu_rom(gg.GELU_SV)
    if x_input_u8 is None:
        image_u8 = gg.load_image_u8(first_existing_path(IMAGE_CANDIDATES, "real input image"))
        x_q, x_scale_calc, patch_shift_calc = _build_patch_input_q(gg, state, image_u8, patch_shift=patch_shift)
        x_scale = x_scale_calc if x_scale is None else x_scale
        patch_shift = patch_shift_calc if patch_shift is None else patch_shift
    else:
        x_q = np.asarray(x_input_u8, dtype=np.uint8).reshape(TOKEN_NUM, EMBED_DIM)
        if x_scale is None:
            x_scale = float(compute_full_vit_integer_golden(force=False, save_cache=True)["x_scale"][0])
    prefix = f"blocks.{int(block_idx)}"
    gamma1_q = np.clip(np.rint(gg.t(state, f"{prefix}.norm1.weight") * (1 << 14)), -32768, 32767).astype(np.int64)
    gamma2_q = np.clip(np.rint(gg.t(state, f"{prefix}.norm2.weight") * (1 << 14)), -32768, 32767).astype(np.int64)
    x_norm = gg.rmsnorm_hw(x_q, gamma1_q, inv_lut)
    qkv_scale = 2.0 ** -4
    qkv_w_scale = qkv_scale / (1 << gg.SHIFT_QKV)
    qkv_w_q, _ = gg.quant_s8(gg.t(state, f"{prefix}.attn.qkv.weight"), qkv_w_scale)
    qkv_b_q = np.rint(gg.t(state, f"{prefix}.attn.qkv.bias") / qkv_w_scale).astype(np.int64)
    q_q, k_q, v_q, _ = gg.qkv_hw(x_norm, qkv_w_q, qkv_b_q)
    q_s = q_q.astype(np.int64) - 128
    k_s = k_q.astype(np.int64) - 128
    v_s = v_q.astype(np.int64) - 128
    scores = np.zeros((gg.HEAD_NUM, TOKEN_NUM, TOKEN_NUM), dtype=np.int64)
    for h in range(gg.HEAD_NUM):
        sl = slice(h * gg.HEAD_DIM, (h + 1) * gg.HEAD_DIM)
        scores[h] = q_s[:, sl] @ k_s[:, sl].T
    attn_q = gg.softmax_hw(scores, exp_lut)[:, :, :TOKEN_NUM]
    o_heads = np.zeros((TOKEN_NUM, EMBED_DIM), dtype=np.uint8)
    for h in range(gg.HEAD_NUM):
        sl = slice(h * gg.HEAD_DIM, (h + 1) * gg.HEAD_DIM)
        o_heads[:, sl] = gg.requant_zp128(attn_q[h].astype(np.int64) @ v_s[:, sl].astype(np.int64), gg.SHIFT_ATTN_V)
    o_scale = (1.0 / 128.0) * qkv_scale * (1 << gg.SHIFT_ATTN_V)
    out_w_scale = x_scale / (o_scale * (1 << gg.SHIFT_OUT_PROJ))
    out_w_q, _ = gg.quant_s8(gg.t(state, f"{prefix}.attn.proj.weight"), out_w_scale)
    out_b_q = np.rint(gg.t(state, f"{prefix}.attn.proj.bias") / (o_scale * out_w_scale)).astype(np.int64)
    out_q, _ = gg.linear_hw(o_heads, out_w_q, out_b_q, gg.SHIFT_OUT_PROJ, act_zp=128)
    x_mid = gg.residual_add(out_q, x_q)
    x_mid_norm = gg.rmsnorm_hw(x_mid, gamma2_q, inv_lut)
    fc1_w_q, _ = gg.quant_s8(gg.t(state, f"{prefix}.mlp.fc1.weight"), float(1.0 / 1024.0))
    fc1_b_q = np.rint(gg.t(state, f"{prefix}.mlp.fc1.bias") / float(1.0 / 1024.0)).astype(np.int64)
    _, fc1_psum = gg.linear_hw(x_mid_norm, fc1_w_q, fc1_b_q, gg.SHIFT_FC1, act_zp=128)
    gelu_q = gg.gelu_requant_hw(fc1_psum, gelu_rom)
    fc2_w_scale = x_scale / (1.0 * (1 << gg.SHIFT_FC2))
    fc2_w_q, _ = gg.quant_s8(gg.t(state, f"{prefix}.mlp.fc2.weight"), fc2_w_scale)
    fc2_b_q = np.rint(gg.t(state, f"{prefix}.mlp.fc2.bias") / fc2_w_scale).astype(np.int64)
    fc2_q, _ = gg.linear_hw(gelu_q, fc2_w_q, fc2_b_q, gg.SHIFT_FC2, act_zp=128)
    x_out = gg.residual_add(fc2_q, x_mid)
    return {"x_after_patch": x_q.astype(np.uint8), "x_norm1": x_norm.astype(np.uint8), "o_attn": o_heads.astype(np.uint8), "x_mid": x_mid.astype(np.uint8), "x_mid_norm": x_mid_norm.astype(np.uint8), "gelu": gelu_q.astype(np.uint8), "x_out": x_out.astype(np.uint8), "x_scale": float(x_scale), "patch_shift": int(patch_shift) if patch_shift is not None else None}

def dump_stage_words(mmio, meta, page_buffers=None, timeout_s=60.0):
    if meta.get("source") == "gelu_ddr_buffer":
        buf = page_buffers["gelu"]
        if hasattr(buf, "invalidate"):
            buf.invalidate()
        return np.asarray(buf, dtype=np.uint32).reshape(-1).copy()
    out = allocate(shape=(int(meta["dump_words"]),), dtype=np.uint32, cacheable=False)
    out[:] = 0
    ddr_store_to_buffer(mmio, target=int(meta["dump_target"]), out=out, word_count=int(meta["dump_words"]), local_base=0, timeout_s=timeout_s)
    if hasattr(out, "invalidate"):
        out.invalidate()
    return np.asarray(out, dtype=np.uint32).reshape(-1).copy()

def rtl_act_word_index(token_idx, channel_idx, channel_num):
    channel_tiles = (int(channel_num) + CHANNEL_TILE - 1) // CHANNEL_TILE
    token_tile_idx = int(token_idx) // TOKEN_TILE
    channel_tile_idx = int(channel_idx) // CHANNEL_TILE
    token_inner = int(token_idx) % TOKEN_TILE
    row_idx = token_tile_idx * channel_tiles * TOKEN_TILE + channel_tile_idx * TOKEN_TILE + token_inner
    lane_idx = (int(channel_idx) >> 2) & (TILE_ROW_WORDS - 1)
    return row_idx * TILE_ROW_WORDS + lane_idx


def extract_activation_phys_words(words, token_num, channel_num):
    words = np.asarray(words, dtype=np.uint32).reshape(-1)
    out = np.empty(int(token_num) * int(channel_num), dtype=np.uint8)
    k = 0
    for token_idx in range(int(token_num)):
        for channel_idx in range(int(channel_num)):
            word_idx = rtl_act_word_index(token_idx, channel_idx, channel_num)
            lane = channel_idx & 3
            if word_idx < len(words):
                out[k] = np.uint8((int(words[word_idx]) >> (8 * lane)) & 0xFF)
            else:
                out[k] = np.uint8(0)
            k += 1
    return out


def compare_stage_dump(stage_id, dump_words, stage_golden, tolerance=0):
    meta = STAGE_META[int(stage_id)]
    key = meta["golden_key"]
    if stage_golden is None or key not in stage_golden:
        return {"skipped": True, "reason": f"missing trusted golden key: {key}"}
    golden_tensor = np.asarray(stage_golden[key], dtype=np.uint8)
    golden = golden_tensor.reshape(-1)
    if meta.get("layout") == "activation_phys":
        channels = int(meta.get("channels", golden_tensor.shape[-1] if golden_tensor.ndim >= 2 else EMBED_DIM))
        token_num = int(meta.get("tokens", TOKEN_NUM))
        rtl = extract_activation_phys_words(dump_words, token_num=token_num, channel_num=channels)
        golden = golden_tensor.reshape(token_num, channels).reshape(-1)
    else:
        rtl = unpack_u32_words_to_u8(dump_words, valid_bytes=meta.get("valid_bytes"))
    return compare_u8_arrays(rtl, golden, tolerance=tolerance)


def print_stage_metrics(stage_name, counters):
    perf = summarize_perf(stage_counters_for_summary(counters), clk_hz=PL_CLK_HZ)
    print(
        f"  metrics: cycles={perf['total_cycles']} busy={perf['compute_busy_cycles']} "
        f"DRAM_cycles={perf['DRAM_access_time_cycles']} BRAM_cycles={perf['BRAM_access_time_cycles']} "
        f"BRAM_BW={perf['BRAM_bandwidth_bytes_per_cycle']:.3f}B/cyc "
        f"DDR_txn={perf['ddr_transactions']} DDR_B/txn={perf['ddr_bytes_per_transaction']:.2f} "
        f"MACs={perf['mac_ops']} "
        f"e2e_MAC/cyc={perf['end_to_end_mac_per_cycle']:.6f} "
        f"sys_run={perf['systolic_run_cycles']} sys_op={perf['systolic_op_cycles']} sys_step={perf['systolic_pe_step_cycles']} "
        f"sysOP_MAC/cyc={perf['systolic_op_mac_per_cycle']:.6f} "
        f"util64={perf['systolic_op_util_vs_rtl_peak']:.6f} util128={perf['systolic_op_util_vs_theory_peak']:.6f} "
        f"pingpong_overlap={perf['pingpong_overlap_ratio']:.3f}"
    )
    return perf


def handle_stage_checkpoint(mmio, stage_status, stage_golden=None, page_buffers=None, tolerance=0, timeout_s=60.0):
    stage_id = int(stage_status["stage_id"])
    meta = STAGE_META.get(stage_id, {"name": f"Unknown stage {stage_id}"})
    counters = read_stage_counters(mmio)
    print(f"\n[stage {stage_id:02d}] {meta['name']} done: seq={stage_status['sequence']} phase={stage_status['phase']}")
    perf = print_stage_metrics(meta["name"], counters)
    dump_words = None
    compare = None
    if stage_golden is not None and stage_id in STAGE_META:
        dump_words = dump_stage_words(mmio, meta, page_buffers=page_buffers, timeout_s=timeout_s)
        compare = compare_stage_dump(stage_id, dump_words, stage_golden, tolerance=tolerance)
        if globals().get("PRINT_ACCURACY_METRICS", False):
            if compare.get("skipped"):
                print(f"  compare: skipped ({compare['reason']})")
            else:
                print(f"  compare: exact={compare['exact_match_percent']:.3f}% within_tol={compare['within_tolerance_percent']:.3f}% max_abs={compare['max_abs_diff']} mean_abs={compare['mean_abs_diff']:.4f} rmse={compare['rmse']:.4f}")
                if compare.get("first_mismatches"):
                    print("  first mismatches:", compare["first_mismatches"][:5])
    return {"stage_id": stage_id, "stage_name": meta["name"], "status": dict(stage_status), "counters": counters, "perf": perf, "compare": compare, "dump_words": dump_words}

def service_tiles_checkpoints_until_done(mmio, patch_weight_words=None, patch_bias_words=None, trans_weight_words=None, trans_bias_words=None, page_buffers=None, stage_golden=None, tolerance=0, timeout_s=900.0, verbose_pages=False, progress_interval_s=30.0, stage_dump_timeout_s=60.0):
    t0 = time.time(); last_progress = t0; last_patch_req = None; last_trans_req = None; stage_results = []
    while True:
        st_word = rd(mmio, REG_STAGE_STATUS); st = decode_stage_status(st_word)
        if st["pending"]:
            stage_results.append(handle_stage_checkpoint(mmio, st, stage_golden=stage_golden, page_buffers=page_buffers, tolerance=tolerance, timeout_s=stage_dump_timeout_s))
            resume_stage_checkpoint(mmio, enable=True); last_progress = time.time(); continue
        if page_buffers is not None and service_page_request_once(mmio, page_buffers, verbose=verbose_pages):
            continue
        status = rd(mmio, REG_STATUS); ds = decode_status(status)
        if ds["done_sticky"]:
            return status, stage_results
        if ds["input_error"]:
            raise RuntimeError(f"Core input_error while running: status=0x{status:08x}, debug={ddr_debug_snapshot(mmio)}")
        patch_req = rd(mmio, REG_PATCH_REQ); trans_req = rd(mmio, REG_TRANS_REQ)
        need_patch = ds["patch_weight_miss"] or ds["patch_bias_miss"]
        need_trans = ds["trans_weight_miss"] or ds["trans_bias_miss"]
        req_changed = (patch_req != last_patch_req) or (trans_req != last_trans_req)
        if need_patch or need_trans:
            load_requested_tiles_once(
                mmio,
                patch_weight_words if ds["patch_weight_miss"] else None,
                patch_bias_words if ds["patch_bias_miss"] else None,
                trans_weight_words if ds["trans_weight_miss"] else None,
                trans_bias_words if ds["trans_bias_miss"] else None,
                verbose=verbose_pages and req_changed,
            )
            last_patch_req, last_trans_req = patch_req, trans_req
            continue
        if req_changed:
            last_patch_req, last_trans_req = patch_req, trans_req
        now = time.time()
        if now - last_progress >= progress_interval_s:
            print_progress_snapshot(mmio, now - t0, ds, patch_req, trans_req) if "print_progress_snapshot" in globals() else print(f"[stage-progress {now - t0:8.1f}s] status=0x{status:08x} stage=0x{st_word:08x}")
            last_progress = now
        if now - t0 > timeout_s:
            raise TimeoutError(f"Timeout waiting for staged run done: status=0x{status:08x}, stage=0x{st_word:08x}, debug={ddr_debug_snapshot(mmio)}")
        time.sleep(0.0005)

def run_one_block_real_model_staged(mmio, patch_shift=None, verify_final=True, tolerance=0, verbose_pages=False, progress_interval_s=30.0, timeout_s=900.0, stage_dump_timeout_s=90.0):
    print("Stage checkpoint driver:", STAGE_CHECKPOINT_DRIVER_VERSION)
    clear_done(mmio); clear_perf(mmio); clear_stage_checkpoint(mmio, enable=True)
    case_manifest = load_case_manifest()
    if patch_shift is None:
        patch_shift = int(case_manifest.get("patch_requant_shift", manifest_patch_shift(0)))
    wr(mmio, REG_PATCH_SHIFT, patch_shift); wr(mmio, REG_RUN_MODE, 0)
    print("Computing software golden for per-stage compare...")
    stage_golden = compute_one_block_stage_golden(block_idx=0, patch_shift=patch_shift)
    static_buffers = load_static_inputs(mmio)
    gelu_buf = alloc_gelu_buffer()
    page_buffers = {"image": static_buffers["image"], "pos": static_buffers["pos"], "gelu": gelu_buf}
    all_words = load_full_design_words(); weight_buffers = alloc_weight_buffers(all_words)
    patch_weight_src = weight_buffers.get("patch_weight"); patch_bias_src = weight_buffers.get("patch_bias")
    trans_weight_src = weight_buffers.get("trans_weight"); trans_bias_src = weight_buffers.get("trans_bias")
    load_requested_tiles_once(mmio, patch_weight_src, patch_bias_src, trans_weight_src, trans_bias_src)
    start_compute(mmio)
    _, stage_results = service_tiles_checkpoints_until_done(mmio, patch_weight_src, patch_bias_src, trans_weight_src, trans_bias_src, page_buffers=page_buffers, stage_golden=stage_golden, tolerance=tolerance, timeout_s=timeout_s, verbose_pages=verbose_pages, progress_interval_s=progress_interval_s, stage_dump_timeout_s=stage_dump_timeout_s)
    enable_stage_checkpoint(mmio, enable=False)
    out_buf = ddr_store_xout(mmio, XOUT_ELEMS, local_base=0, timeout_s=stage_dump_timeout_s)
    counters = read_perf_counters(mmio)
    verify_result = verify_xout(out_buf, tolerance=tolerance) if verify_final else None
    print("\nPer-stage summary")
    for r in stage_results:
        c = r.get("compare")
        if globals().get("PRINT_ACCURACY_METRICS", False):
            print(f"  stage {r['stage_id']:02d} {r['stage_name']}: cycles={r['perf']['total_cycles']} exact={(c or {}).get('exact_match_percent', 0):.3f}% within±{tolerance}={(c or {}).get('within_tolerance_percent', 0):.3f}%")
        else:
            print(f"  stage {r['stage_id']:02d} {r['stage_name']}: cycles={r['perf']['total_cycles']} DRAM_cycles={r['perf'].get('dram_access_time_cycles', 0)} BRAM_cycles={r['perf'].get('bram_access_time_cycles', 0)}")
    run_one_block_real_model_staged.last_stage_results = stage_results
    run_one_block_real_model_staged.last_stage_golden = stage_golden
    run_one_block_real_model_staged.last_buffers = {"static": static_buffers, "gelu": gelu_buf, "weights": weight_buffers}
    run_one_block_real_model_staged.last_counters = counters
    run_one_block_real_model_staged.last_verify = verify_result
    return out_buf, counters, verify_result, stage_results


## Run One Block With Per-stage Checkpoints

Run this cell after loading the overlay/MMIO. It pauses at every RTL checkpoint, prints metrics, dumps the stage buffer, compares against software golden, then resumes the hardware.


In [13]:
# Robust staged run cell. This cell reloads all definition cells first so a stale
# Jupyter kernel cannot miss helpers such as ddr_load/ddr_store/load_static_inputs.
def _vit_reload_definition_cells():
    import json as _json
    from pathlib import Path as _Path
    nb_path = _Path.cwd() / "vit.ipynb"
    if not nb_path.exists():
        raise FileNotFoundError(f"Cannot find notebook for bootstrap: {nb_path}")
    nb = _json.loads(nb_path.read_text(encoding="utf-8"))
    markers = [
        "from pathlib import Path",
        "REG_CONTROL      =",
        "def load_overlay",
        "def wr(mmio",
        "def split_hex_token_to_u32_words",
        "def _ddr_error_from_status",
        "IMG_H = 224",
        "def alloc_gelu_buffer",
        "def read_perf_counters",
        "VIVADO_CORE_DYNAMIC_POWER_W",
        "FULL_VIT_BLOCKS = 12",
        "STAGE_CHECKPOINT_DRIVER_VERSION",
    ]
    used_cells = []
    for marker in markers:
        found = False
        for idx, cell in enumerate(nb.get("cells", [])):
            if cell.get("cell_type") != "code":
                continue
            src = "".join(cell.get("source", []))
            if marker not in src:
                continue
            # Do not execute this run cell recursively.
            if "_vit_reload_definition_cells" in src:
                continue
            exec(compile(src, f"{nb_path}:cell{idx}", "exec"), globals())
            used_cells.append(idx)
            found = True
            break
        if not found:
            raise RuntimeError(f"Notebook bootstrap cannot find definition cell containing marker: {marker}")
    required = [
        "load_overlay", "wr", "rd", "decode_status", "decode_ddr_status",
        "ddr_load", "ddr_load_addr", "ddr_store_addr", "ddr_store_to_buffer",
        "load_static_inputs", "alloc_gelu_buffer", "load_full_design_words",
        "alloc_weight_buffers", "load_requested_tiles_once", "start_compute",
        "service_tiles_checkpoints_until_done", "run_one_block_real_model_staged",
        "read_perf_counters", "verify_xout",
    ]
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError("Notebook bootstrap is incomplete; missing: " + ", ".join(missing))
    print("Bootstrap definition cells:", used_cells)

_vit_reload_definition_cells()
print("Notebook driver:", DRIVER_VERSION)
print("Stage checkpoint driver:", STAGE_CHECKPOINT_DRIVER_VERSION)

# Always reprogram before the staged run. Previous timeouts/checkpoints can leave RTL busy,
# so this clears hardware state and avoids stale MMIO/core status.
overlay, mmio = load_overlay()

out_buf, counters, verify_result, stage_results = run_one_block_real_model_staged(
    mmio,
    verify_final=True,
    tolerance=0,
    verbose_pages=False,
    progress_interval_s=30.0,
    timeout_s=900.0,
    stage_dump_timeout_s=90.0,
)
print("Final counters:", counters)
print("Final verify:", verify_result)
stage_results


Notebook driver: 2026-06-25-stage-checkpoint-v18-rtl-physical-golden
Notebook dir: /home/xilinx/jupyter_notebooks/VIT_fulldesign
Selected golden root: /home/xilinx/jupyter_notebooks/VIT_fulldesign/golden_gen/hex/case_vit_real_model
Bootstrap definition cells: [1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 20, 22]
Notebook driver: 2026-06-25-stage-checkpoint-v18-rtl-physical-golden
Stage checkpoint driver: 2026-06-25-stage-checkpoint-v18-rtl-physical-golden
Using IP ViT_DDR_AxiLite_Wrap_0: base=0x40000000, range=0x100000
Stage checkpoint driver: 2026-06-25-stage-checkpoint-v18-rtl-physical-golden
Computing software golden for per-stage compare...
Image DDR buffer: 37632 words @ 0x16880000
Position DDR buffer: 18912 words @ 0x16860000
Loading cls words into RTL: 96
Loading gamma words into RTL: 768
GELU_out DDR buffer: 76800 words @ 0x16900000
patch_weight: 73728 words from /home/xilinx/jupyter_notebooks/VIT_fulldesign/golden_gen/hex/case_vit_real_model/patch_weight.hex
patch_bias: 384 words from /

KeyboardInterrupt: 

## Presentation-normalized Optimized INT8 profiling table

This cell is a model/projection table for the report. It does **not** replace RTL hardware counters above.
It uses the profiling assumptions from the optimized INT8 model to reproduce the presentation-style full one-block group table.


In [15]:
# Presentation-normalized Optimized INT8 profiling table
# This is a theoretical/model projection for report comparison, not a replacement
# for measured RTL counters.  The clock here follows the profiling table:
# 1.682M cycles / 100 MHz = 16.82 ms.

NORMALIZED_PROFILE_CLK_HZ = 100_000_000
PROFILE_MAC_ENERGY_J = 1.0e-12       # 1 pJ / MAC
PROFILE_BRAM_ENERGY_J_PER_BYTE = 5.0e-12
PROFILE_DRAM_ENERGY_J_PER_BYTE = 200.0e-12
PROFILE_LEAKAGE_POWER_W = 0.200065   # rounds to 0.2 W in the profiling assumption


def _fmt_millions(x):
    return f"{x / 1_000_000:.3f}M".rstrip("0").rstrip(".") + "M" if False else f"{x / 1_000_000:.3f}M"


def optimized_int8_projection_row():
    cycles = 1_682_000
    performance = 239.03
    latency_ms = cycles / NORMALIZED_PROFILE_CLK_HZ * 1e3
    macs = cycles * performance
    dram_mb = 1.785
    bram_mb = 53.78
    dram_bytes = dram_mb * 1_000_000
    bram_bytes = bram_mb * 1_000_000
    compute_energy_j = macs * PROFILE_MAC_ENERGY_J
    bram_energy_j = bram_bytes * PROFILE_BRAM_ENERGY_J_PER_BYTE
    dram_energy_j = dram_bytes * PROFILE_DRAM_ENERGY_J_PER_BYTE
    leakage_energy_j = PROFILE_LEAKAGE_POWER_W * (latency_ms / 1e3)
    energy_uj = (compute_energy_j + bram_energy_j + dram_energy_j + leakage_energy_j) * 1e6
    return {
        "model": "Optimized INT8 projection",
        "cycles": f"{cycles / 1_000_000:.3f}M",
        "latency": f"{latency_ms:.2f} ms",
        "performance": f"{performance:.2f} MAC/cycle",
        "DRAM total": f"{dram_mb:.3f} MB",
        "BRAM total": f"{bram_mb:.2f} MB",
        "energy": f"{energy_uj:,.0f} uJ",
        "bound": "compute",
        "macs": macs,
        "energy_breakdown_uJ": {
            "compute": compute_energy_j * 1e6,
            "BRAM": bram_energy_j * 1e6,
            "DRAM": dram_energy_j * 1e6,
            "leakage": leakage_energy_j * 1e6,
        },
    }


def baseline_fp32_projection_row():
    return {
        "model": "Baseline-B FP32 projection",
        "cycles": "22.11M",
        "latency": "221.14 ms",
        "performance": "18.30 MAC/cycle",
        "DRAM total": "215.60 MB",
        "BRAM total": "0 MB",
        "energy": "88,967 uJ",
        "bound": "memory",
    }


def print_projection_markdown_table(rows):
    cols = ["model", "cycles", "latency", "performance", "DRAM total", "BRAM total", "energy", "bound"]
    print("| " + " | ".join(cols) + " |")
    print("|" + "|".join(["---"] + ["---:"] * (len(cols) - 2) + ["---"]) + "|")
    for r in rows:
        print("| " + " | ".join(str(r[c]) for c in cols) + " |")


projection_rows = [baseline_fp32_projection_row(), optimized_int8_projection_row()]
print_projection_markdown_table(projection_rows)
print("\nOptimized INT8 energy breakdown (uJ):")
for k, v in optimized_int8_projection_row()["energy_breakdown_uJ"].items():
    print(f"  {k}: {v:,.2f}")


| model | cycles | latency | performance | DRAM total | BRAM total | energy | bound |
|---|---:|---:|---:|---:|---:|---:|---|
| Baseline-B FP32 projection | 22.11M | 221.14 ms | 18.30 MAC/cycle | 215.60 MB | 0 MB | 88,967 uJ | memory |
| Optimized INT8 projection | 1.682M | 16.82 ms | 239.03 MAC/cycle | 1.785 MB | 53.78 MB | 4,393 uJ | compute |

Optimized INT8 energy breakdown (uJ):
  compute: 402.05
  BRAM: 268.90
  DRAM: 357.00
  leakage: 3,365.09
